# Residual-weighted ensemble Kalman–particle filtering for behavioral COVID-19 modeling and retrospective forecasting

Run **Restart Kernel → Run All** to review the saved numerical results and regenerate publication figures from those saved inputs. This notebook performs no model fitting or forecast simulations.

**Current scope:** Main Tables 1–4 and Figures 1–7; Supplementary Tables S1–S20 and Figures S1–S7.

- Full-wave fitting, diagnostics, behavioral scenarios, and sensitivity retain Korea, Japan, and Pakistan.
- Completed peak forecasts retain Korea, Japan, and Pakistan, with six active/recovered outcomes. Pakistan's training initializer reaches its evaluation budget (exit flag 0; 1,501 evaluations); this is disclosed rather than treated as failed forecast integration. Korea/Japan forms a separate converged-initializer robustness subset.
- Main Figure 2 restores both EnKF and EnKPF on observed-scale linear axes. Figure S7 shows their complete means, latent intervals, and member ranges without clipping.
- Fourteen-day summaries retain all five declared computational seeds at all three origins (21/30 favorable EnKPF comparisons); the Korea/Japan robustness subset is 16/20. Separate completed exploratory limitations remain disclosed in the manuscripts; unresolved profiles and incomplete extensions are not displayed as validated results here.
- Tables read cached source CSVs. `EXPORT_NUMBERED_TABLES = True` writes the current 24-table workbook and CSV/TeX fragments. This does not run MATLAB or a forecast model.
- Existing fitting, policy, parameter, and PRCC plots are displayed by default. `REBUILD_EXISTING_FIGURES = True` redraws those plots from saved results only.

Keep the project folders together, including `Python_2026`, `AMM_Validation/editorial`, and the saved result CSVs. Dependencies are `numpy`, `pandas`, `scipy>=1.15`, `matplotlib>=3.10`, `openpyxl`, `jinja2`, `Pillow`, and `IPython`. The final cell can regenerate both restored forecast figures directly from the cached native MATLAB exports.

All required Python helper implementation is embedded below. No MATLAB code is included; this notebook reviews and formats saved exports and does not reproduce their fitting or filtering computations.


## 1. Packages

In [ ]:
import sys
import json
import importlib
import types
from typing import Any
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import matplotlib
import matplotlib.pyplot as plt
import openpyxl
from PIL import Image as PILImage
from IPython.display import display, Markdown, Image as NotebookImage

## 2. Portable package directories
Open Jupyter inside this package, or set PACKAGE_ROOT_OVERRIDE. Every input and output path is relative to this folder.


In [ ]:
PACKAGE_ROOT_OVERRIDE: str | Path | None = None

def resolve_package_root(override: str | Path | None = None) -> Path:
    candidates = [Path(override).expanduser().resolve()] if override is not None else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "Data/Source_Manifest.json").is_file() and (candidate / "Figures").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Open Jupyter inside EnKPF_Publication_Package, or set PACKAGE_ROOT_OVERRIDE.")

PROJECT_ROOT = resolve_package_root(PACKAGE_ROOT_OVERRIDE)
PYTHON_DIR = PROJECT_ROOT
COUNTRIES = {"South Korea": PROJECT_ROOT / "Data/Korea", "Japan": PROJECT_ROOT / "Data/Japan", "Pakistan": PROJECT_ROOT / "Data/Pak"}
RESULTS_DIR = PROJECT_ROOT / "Results"
PNG_DIR = PROJECT_ROOT / "Figures/PNG"
TIFF_DIR = PROJECT_ROOT / "Figures/TIFF"
TABLES_DIR = RESULTS_DIR / "Tables"
TEX_DIR = RESULTS_DIR / "manuscript_tables"
VALIDATION_DIR = PROJECT_ROOT / "Data/Validation"
EDITORIAL_DIR = PROJECT_ROOT / "Results/Forecast"
for directory in (PNG_DIR, TIFF_DIR, TABLES_DIR, TEX_DIR, EDITORIAL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

EMBEDDED_MODULE_SOURCES = {'publication_policy': '"""Read scenario settings from saved policy results, without relabeling them."""\nimport numpy as np\n\n\ndef policy_protocol(waves, source="policy exports"):\n    """Require one matched protocol across all supplied country-wave rows.\n\n    Activation fractions and magnitudes are inferred from the saved rows.\n    A partially rerun set of countries must not be combined into one contrast.\n    """\n    required = {"Wave_Duration_Days", "Early_Activation_Elapsed_Days",\n                "Late_Activation_Elapsed_Days", "Early_BetaF_Change_Percent",\n                "Late_BetaF_Change_Percent"}\n    if waves.empty or not required.issubset(waves.columns):\n        raise ValueError(f"{source}: policy settings are missing or empty.")\n    duration = waves["Wave_Duration_Days"].to_numpy(dtype=float)\n    if not np.isfinite(duration).all() or (duration <= 0).any():\n        raise ValueError(f"{source}: wave durations must be positive and finite.")\n    settings = {}\n    for name in ("Early", "Late"):\n        fraction = waves[f"{name}_Activation_Elapsed_Days"].to_numpy(dtype=float) / duration\n        change = waves[f"{name}_BetaF_Change_Percent"].to_numpy(dtype=float)\n        if (not np.isfinite(fraction).all() or (fraction < 0).any() or (fraction > 1).any()\n                or not np.allclose(fraction, fraction[0], rtol=1e-10, atol=1e-12)):\n            raise ValueError(f"{source}: inconsistent or invalid {name.lower()} activation fractions. Rerun all three policy_G.m scripts with the same settings.")\n        if not np.isfinite(change).all() or (change <= 0).any() or not np.allclose(change, change[0], rtol=1e-10, atol=1e-12):\n            raise ValueError(f"{source}: inconsistent {name.lower()} magnitudes. Rerun all three policy_G.m scripts with the same settings.")\n        settings[f"{name.lower()}_percent"] = float(change[0])\n        settings[f"{name.lower()}_fraction"] = float(fraction[0])\n        settings[f"{name.lower()}_label"] = f"{name}\\n+{change[0]:g}% at {100*fraction[0]:g}%"\n    if settings["early_fraction"] >= settings["late_fraction"]:\n        raise ValueError(f"{source}: early activation must precede late activation.")\n    same = bool(np.isclose(settings["early_percent"], settings["late_percent"], rtol=1e-10, atol=1e-12))\n    settings["timing_only"] = same\n    settings["description"] = (\n        f"Early: +{settings[\'early_percent\']:g}% beta_f at {100*settings[\'early_fraction\']:g}% of elapsed wave duration; "\n        f"late: +{settings[\'late_percent\']:g}% at {100*settings[\'late_fraction\']:g}%."\n    )\n    settings["interpretation"] = (\n        "The perturbation magnitude is held fixed, so this contrast isolates activation timing within the fitted model. "\n        if same else "Timing and perturbation magnitude both differ, so this contrast does not isolate timing. "\n    ) + "These conditional simulations do not estimate a causal policy effect."\n    return settings\n', 'publication_styles': '"""Consistent final-width exports for the notebook\'s scientific figures."""\nfrom pathlib import Path\nimport textwrap\n\nimport matplotlib.dates as mdates\nimport matplotlib.pyplot as plt\nfrom matplotlib.text import Text\nfrom matplotlib.ticker import MaxNLocator, FixedLocator\nfrom PIL import Image\n\n\nCOUNTRY_DISPLAY = {"South Korea": "Korea", "Japan": "Japan", "Pakistan": "Pakistan"}\n\n\ndef country_display(country):\n    return COUNTRY_DISPLAY.get(country, country)\n\n\ndef save_publication_figure(fig, filename, png_dir, tiff_dir, dpi=600):\n    stem = Path(filename).stem\n    sizes = {\n        "graphical_abstract": (7, 6.2),\n        "Combined_3x2_Panel_Fitting": (7, 7.0),\n        "Figure3_EnKPF_Behavioral_Parameter_Trajectories": (7, 6.6),\n        "Figure4_Behavioral_Parameters_By_Wave_Country_PercentChange": (7, 9.3),\n        "Figure_Global_LHS_PRCC_QR": (7, 3.8),\n    }\n    target = sizes.get(stem)\n    if stem.startswith("Parameter_Comparison_"):\n        target = (7, 7.8)\n    if target:\n        original_width = fig.get_size_inches()[0]\n        factor = target[0] / original_width\n        fig.set_size_inches(*target, forward=True)\n        for item in fig.findobj(match=Text):\n            item.set_fontsize(max(9, min(11, item.get_fontsize()*factor)))\n        if fig._suptitle is not None:\n            fig._suptitle.set_text("")\n        handles, labels = [], []\n        if fig.legends:\n            legend = fig.legends[0]\n            handles = list(legend.legend_handles)\n            labels = [item.get_text() for item in legend.get_texts()]\n        elif fig.axes:\n            handles, labels = fig.axes[0].get_legend_handles_labels()\n        for legend in list(fig.legends):\n            legend.remove()\n        for ax in fig.axes:\n            if ax.get_legend() is not None:\n                ax.get_legend().remove()\n            if len(ax.get_title()) > 38:\n                ax.set_title(textwrap.fill(ax.get_title(), 35), fontsize=10, fontweight="bold")\n            for title in (ax.title, ax._left_title, ax._right_title):\n                if title.get_text():\n                    title.set_fontsize(10)\n                    title.set_fontweight("bold")\n            for line in ax.get_lines():\n                if line.get_linewidth() > 1.6:\n                    line.set_linewidth(1.3)\n                if line.get_markersize() > 4:\n                    line.set_markersize(4)\n            categorical_mesh = any(type(item).__name__ == "QuadMesh" for item in ax.collections)\n            if ax.name == "rectilinear" and not categorical_mesh and ax.get_label() != "<colorbar>":\n                ax.yaxis.set_major_locator(MaxNLocator(nbins=3, min_n_ticks=2))\n                if isinstance(ax.xaxis.get_major_locator(), mdates.DateLocator):\n                    ax.xaxis.set_major_locator(FixedLocator(ax.get_xlim())\n                        if stem.startswith("Parameter_Comparison_")\n                        else mdates.AutoDateLocator(minticks=2, maxticks=4))\n                    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))\n            ax.tick_params(labelsize=9)\n        if handles:\n            fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(.5, .005),\n                ncol=min(3, len(labels)), fontsize=9, frameon=False)\n        if stem.startswith("Figure4_"):\n            # Manual spacing keeps nine short time-series rows tall enough to read.\n            for index, ax in enumerate(fig.axes):\n                for item in ax.findobj(match=Text):\n                    item.set_fontsize(10.5)\n                ax.set_xlabel("Reporting date" if index >= 16 else "", fontsize=10.5)\n                ax.tick_params(axis="both", labelsize=10.5)\n                ax.tick_params(axis="x", labelbottom=True)\n                ax.xaxis.set_major_locator(FixedLocator(ax.get_xlim()))\n                ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b %y"))\n                ax.set_ylabel(r"$\\beta_f$" if index % 2 == 0 else r"$\\mu_f$", fontsize=10.5)\n            fig.subplots_adjust(left=.12, right=.98, bottom=.06, top=.97,\n                hspace=.95, wspace=.55)\n        else:\n            fig.tight_layout(rect=(0, .065 if handles else .005, 1, .995),\n                h_pad=1.3, w_pad=1)\n    png_dir, tiff_dir = Path(png_dir), Path(tiff_dir)\n    png_dir.mkdir(parents=True, exist_ok=True)\n    tiff_dir.mkdir(parents=True, exist_ok=True)\n    png = png_dir / f"{stem}.png"\n    fig.savefig(png, dpi=max(300, dpi), facecolor="white", bbox_inches="tight")\n    fig.savefig(png_dir / f"{stem}.pdf", facecolor="white", bbox_inches="tight")\n    with Image.open(png) as raster:\n        raster.convert("RGB").save(tiff_dir / f"{stem}.tif",\n            compression="tiff_lzw", dpi=(max(300, dpi), max(300, dpi)))\n    return png\n\n\ndef behavioral_parameter_overview(parameter_wide, save_figure):\n    fig, axes = plt.subplots(3, 2, figsize=(7, 6.6))\n    for row, country in enumerate(("South Korea", "Japan", "Pakistan")):\n        for col, (parameter, label) in enumerate((("beta_f", r"$\\beta_f$"), ("mu_f", r"$\\mu_f$"))):\n            ax = axes[row, col]\n            for wave in (1, 2, 3):\n                series = parameter_wide.loc[parameter_wide.Country.eq(country)\n                    & parameter_wide.Parameter.eq(parameter)\n                    & parameter_wide.Wave.eq(f"Wave {wave}")].sort_values("Date")\n                if series.empty:\n                    raise ValueError(f"Missing {country} Wave {wave} {parameter}.")\n                ax.axvspan(series.Date.iloc[0], series.Date.iloc[-1],\n                    color="#e8eef4" if wave % 2 else "#f6f6f6", zorder=0)\n                ax.plot(series.Date, series.EnKPF, color="#0072B2", lw=1.3)\n                if wave > 1:\n                    ax.axvline(series.Date.iloc[0], color="#777777", ls="--", lw=.7)\n            ax.set_title(f"{country_display(country)}: {label}", fontsize=10, fontweight="bold")\n            ax.set_ylabel(label, fontsize=10)\n            ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=2, maxticks=3))\n            ax.grid(axis="y", alpha=.2)\n            for side in ("top", "right"):\n                ax.spines[side].set_visible(False)\n            if row == 2:\n                ax.set_xlabel("Reporting date", fontsize=9)\n    save_figure(fig, "Figure3_EnKPF_Behavioral_Parameter_Trajectories", dpi=600)\n    plt.show()\n    plt.close(fig)\n', 'manuscript_publication': '"""Current-source data and figure builders for the main and SI notebooks.\n\nAll workbook rows keep their wave identity, including shared boundary dates.\nThis module formats the saved experiments; it does not fit a forecasting model.\n"""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\n\nimport matplotlib.dates as mdates\nimport matplotlib.pyplot as plt\nfrom matplotlib.lines import Line2D\nfrom matplotlib.ticker import FixedLocator, MaxNLocator\nimport numpy as np\nimport pandas as pd\nfrom publication_policy import policy_protocol\nfrom publication_styles import country_display\n\n\nCOUNTRY_ORDER = ("South Korea", "Japan", "Pakistan")\nPREFIXES = {"South Korea": "Korea", "Japan": "Jap", "Pakistan": "Pak"}\nDATE_CONFIG = {\n    "South Korea": ("Coviddata-korea1.xlsx", "Rec_Acc", ((27, 204), (204, 296), (296, 391))),\n    "Japan": ("covid_data-japan1.xlsx", "RecAcc", ((50, 160), (160, 275), (275, 391))),\n    "Pakistan": ("covid_data-pak1.xlsx", "RecAcc", ((20, 230), (230, 375), (375, 480))),\n}\nPARAMETERS = ("beta", "beta_f", "mu_f", "I_0")\nPARAMETER_LABELS = {"beta": r"$\\beta$", "beta_f": r"$\\beta_f$", "mu_f": r"$\\mu_f$", "I_0": r"$I_0$"}\nMETHOD_COLORS = {"EnKF": "#1764a0", "EnKPF": "#c44b20"}\nCOUNTRY_COLORS = {"South Korea": "#0072B2", "Japan": "#D55E00", "Pakistan": "#009E73"}\nFIGURES = [\n    ("Main", "1", "Observed active and recovered counts with fitted moment bands", "Combined_3x2_Panel_Fitting"),\n    ("Main", "2", "Behavioral parameters and explicitly timed scenarios", "graphical_abstract"),\n    ("Main", "3", "Date-resolved EnKPF behavioral parameters", "Figure3_EnKPF_Behavioral_Parameter_Trajectories"),\n    ("Main", "4", "Within-wave EnKPF parameter changes", "Figure4_Behavioral_Parameters_By_Wave_Country_PercentChange"),\n    ("Main", "5", "One common-design global LHS–PRCC analysis", "Figure_Global_LHS_PRCC_QR"),\n    ("Main", "6", "Retrospective peak-cutoff forecasts and ensemble intervals", "Figure6_Peak_Cutoff_Forecasts"),\n    ("Supplementary", "S1", "Endpoint Jarque–Bera and Mardia p-values", "Supplementary_Figure_1_p_values"),\n    ("Supplementary", "S2", "Endpoint marginal skewness", "Supplementary_Figure_2_Skewness"),\n    ("Supplementary", "S3", "Endpoint marginal excess kurtosis", "Supplementary_Figure_3_Kurtosis"),\n    ("Supplementary", "S4", "South Korea parameter comparisons", "Parameter_Comparison_South_Korea_WaveWise"),\n    ("Supplementary", "S5", "Japan parameter comparisons", "Parameter_Comparison_Japan_WaveWise"),\n    ("Supplementary", "S6", "Pakistan parameter comparisons", "Parameter_Comparison_Pakistan_WaveWise"),\n    ("Supplementary", "S7", "Peak forecasts with the complete ensemble range", "FigureS_Peak_Cutoff_FullRange"),\n]\n\n\ndef _require_columns(frame, columns, source):\n    missing = set(columns) - set(frame.columns)\n    if missing or frame.empty:\n        raise ValueError(f"{source}: missing columns {sorted(missing)} or empty data.")\n\n\ndef _source_record(path, project_root):\n    path = Path(path)\n    stat = path.stat()\n    return {"Source": path.relative_to(project_root).as_posix(), "Bytes": stat.st_size,\n        "Modified_UTC": datetime.fromtimestamp(stat.st_mtime, timezone.utc).isoformat(),\n        "SHA256": hashlib.sha256(path.read_bytes()).hexdigest()}\n\n\ndef load_publication_data(project_root, countries):\n    """Load matched fitting, parameter and policy sources without cached tables."""\n    project_root = Path(project_root)\n    time_frames, parameter_rows, policy_frames, policy_parameters, impact_frames = [], [], [], [], []\n    rmse_frames, criterion_frames, sources = [], [], []\n    for country in COUNTRY_ORDER:\n        country_dir = Path(countries[country])\n        prefix = PREFIXES[country]\n        workbook = country_dir / "Results" / "Results_EnKPF_LSM" / f"{prefix}_complete_results.xlsx"\n        excel_name, recovered, ranges = DATE_CONFIG[country]\n        observation_path = project_root / "Data" / excel_name\n        raw = pd.read_excel(observation_path)\n        _require_columns(raw, ("date", "Active_cases", recovered), observation_path)\n        raw["date"] = pd.to_datetime(raw["date"], errors="raise")\n        ts = pd.read_excel(workbook, sheet_name="TimeSeries_Fitting")\n        pm = pd.read_excel(workbook, sheet_name="Dynamic_Parameters")\n        expected = pd.concat([raw.iloc[first-1:stop] for first, stop in ranges], ignore_index=True)\n        _require_columns(ts, ("Date", "Obs_Active", "Obs_Recovered", "EnKF_Active", "EnKPF_Active"), workbook)\n        _require_columns(pm, ("Date", *[f"{method}_{p}" for p in ("beta", "beta_f", "mu_f", "I0") for method in METHOD_COLORS]), workbook)\n        for frame in (ts, pm):\n            frame["Date"] = pd.to_datetime(frame["Date"], errors="raise")\n            if len(frame) != len(expected) or not np.array_equal(frame.Date.to_numpy(), expected.date.to_numpy()):\n                raise ValueError(f"{country}: saved result dates disagree with the exact country wave windows.")\n            frame["Country"] = country\n        for column, observed_column in (("Obs_Active", "Active_cases"), ("Obs_Recovered", recovered)):\n            if not np.allclose(ts[column].to_numpy(float), expected[observed_column].to_numpy(float), rtol=0, atol=1e-8):\n                raise ValueError(f"{country}: saved {column} observations disagree with source Excel.")\n        offset = 0\n        for wave, (first, stop) in enumerate(ranges, 1):\n            length = stop - first + 1\n            pm_wave = pm.iloc[offset:offset+length]\n            ts.loc[offset:offset+length-1, "Wave"] = f"Wave {wave}"\n            for parameter in PARAMETERS:\n                column_name = "I0" if parameter == "I_0" else parameter\n                values = pm_wave[[f"{method}_{column_name}" for method in METHOD_COLORS]].to_numpy(float)\n                if not np.isfinite(values).all() or (values < 0).any():\n                    raise ValueError(f"{country} Wave {wave}: invalid parameter estimates.")\n                parameter_rows.extend({"Country": country, "Wave": f"Wave {wave}", "Date": date,\n                    "Parameter": parameter, "EnKF": float(v[0]), "EnKPF": float(v[1])}\n                    for date, v in zip(pm_wave.Date, values))\n            offset += length\n        time_frames.append(ts)\n        for sheet, frames in (("RMSE_Summary", rmse_frames), ("AIC_Summary", criterion_frames)):\n            frame = pd.read_excel(workbook, sheet_name=sheet)\n            frame["Country"] = country\n            frames.append(frame)\n        policy_dir = country_dir / "Results" / "Policy" / "xlsx"\n        wave_path = policy_dir / "EnKPF_G_wave_policy_summary.csv"\n        param_path = policy_dir / "EnKPF_G_policy_parameters.csv"\n        impact_path = policy_dir / "EnKPF_G_policy_impact_summary.csv"\n        waves, params, impact = pd.read_csv(wave_path), pd.read_csv(param_path), pd.read_csv(impact_path)\n        _require_columns(waves, ("Wave", "Wave_Duration_Days", "Early_Activation_Elapsed_Days", "Late_Activation_Elapsed_Days", "Early_BetaF_Change_Percent", "Late_BetaF_Change_Percent", "Early_Reduction_Percent", "Late_Reduction_Percent"), wave_path)\n        if sorted(waves.Wave.to_list()) != [1, 2, 3] or sorted(params.Wave.to_list()) != [1, 2, 3]:\n            raise ValueError(f"{country}: policy tables require waves 1, 2 and 3 exactly once.")\n        waves = waves.sort_values("Wave")\n        policy_protocol(waves, str(wave_path))\n        for frame in (waves, params, impact):\n            frame["Country"] = country\n        for scenario, column in (("Early Intervention", "Early_Reduction_Percent"),\n                                 ("Late Intervention", "Late_Reduction_Percent"),\n                                 ("No Change", "NoChange_Reduction_Percent")):\n            selected = impact.loc[impact.Scenario.eq(scenario), "Average_Peak_Reduction_Percent"]\n            if len(selected) != 1 or not np.isclose(selected.iloc[0], waves[column].mean(), rtol=1e-8, atol=1e-8):\n                raise ValueError(f"{country}: {scenario} averages disagree between policy summaries.")\n        policy_frames.append(waves); policy_parameters.append(params); impact_frames.append(impact)\n        sources.extend(_source_record(p, project_root) for p in (workbook, observation_path, wave_path, param_path, impact_path))\n    result = {"timeseries": pd.concat(time_frames, ignore_index=True),\n        "parameter_wide": pd.DataFrame(parameter_rows), "policy_waves": pd.concat(policy_frames, ignore_index=True),\n        "policy_parameters": pd.concat(policy_parameters, ignore_index=True), "policy_impacts": pd.concat(impact_frames, ignore_index=True),\n        "hybrid_rmse": pd.concat(rmse_frames, ignore_index=True), "hybrid_criterion": pd.concat(criterion_frames, ignore_index=True),\n        "sources": pd.DataFrame(sources)}\n    result["policy_protocol"] = policy_protocol(result["policy_waves"])\n    # A policy rerun must use the same fitted parameter trajectories as the figures.\n    means = result["parameter_wide"].groupby(["Country", "Wave", "Parameter"], sort=False).EnKPF.mean()\n    for row in result["policy_parameters"].itertuples(index=False):\n        for parameter, field in (("beta", "Beta"), ("beta_f", "Beta_f"), ("mu_f", "Mu_f"), ("I_0", "I0")):\n            if not np.isclose(means.loc[(row.Country, f"Wave {row.Wave}", parameter)], getattr(row, field), rtol=1e-7, atol=1e-10):\n                raise ValueError(f"{row.Country} Wave {row.Wave}: policy parameters are not matched to the current hybrid workbook.")\n    return result\n\n\ndef _axis_style(ax):\n    ax.tick_params(labelsize=9)\n    ax.yaxis.set_major_locator(MaxNLocator(3))\n    ax.grid(axis="y", color="#d9dee5", lw=.5)\n    ax.set_axisbelow(True)\n    for side in ("top", "right"):\n        ax.spines[side].set_visible(False)\n\n\ndef _date_style(ax):\n    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=2, maxticks=4))\n    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))\n\n\ndef plot_fitted_trajectories(data):\n    fig, axes = plt.subplots(3, 2, figsize=(7, 7))\n    for row, country in enumerate(COUNTRY_ORDER):\n        frame = data["timeseries"].loc[lambda x: x.Country.eq(country)]\n        for col, (outcome, observed, band) in enumerate((("Active", "Obs_Active", "Active"), ("Recovered", "Obs_Recovered", "Rec"))):\n            ax = axes[row, col]\n            ax.plot(frame.Date, frame[observed], "k.", ms=2, alpha=.7, label="Observed")\n            for method, color in METHOD_COLORS.items():\n                ax.fill_between(frame.Date, frame[f"{method}_{band}_CI_Low"], frame[f"{method}_{band}_CI_High"], color=color, alpha=.17, label=f"{method} ±1.96 SD")\n                ax.plot(frame.Date, frame[f"{method}_{outcome}"], color=color, lw=1.1, label=method)\n            ax.set_title(f"{country_display(country)}: {\'active Q\' if col == 0 else \'recovered R\'}", fontsize=10, fontweight="bold")\n            ax.set_ylabel("Cases", fontsize=9)\n            if row == 2: ax.set_xlabel("Reporting date", fontsize=9)\n            _axis_style(ax); _date_style(ax)\n    return fig\n\n\ndef plot_behavioral_scenarios(data):\n    fig, axes = plt.subplots(2, 2, figsize=(7, 6.2))\n    protocol = policy_protocol(data["policy_waves"])\n    params = data["policy_parameters"]\n    for col, (name, label) in enumerate((("Beta_f", r"$\\beta_f$"), ("Mu_f", r"$\\mu_f$"))):\n        ax = axes[0, col]\n        for country in COUNTRY_ORDER:\n            frame = params.loc[params.Country.eq(country)].sort_values("Wave")\n            ax.plot(frame.Wave, frame[name], "o-", ms=4, lw=1.3, color=COUNTRY_COLORS[country], label=country_display(country))\n        ax.set_title(f"{\'A\' if col == 0 else \'B\'}) Wave-time mean {label}", fontsize=10, fontweight="bold")\n        ax.set_xticks([1, 2, 3], ["Wave 1", "Wave 2", "Wave 3"])\n        ax.set_ylabel(label + r" (day$^{-1}$)", fontsize=9)\n        _axis_style(ax)\n    ax = axes[1, 0]\n    for index, country in enumerate(COUNTRY_ORDER):\n        frame = data["policy_impacts"].loc[lambda x: x.Country.eq(country)].set_index("Scenario")\n        values = [frame.loc[scenario, "Average_Peak_Reduction_Percent"] for scenario in ("Early Intervention", "Late Intervention", "No Change")]\n        values = np.asarray(values, dtype=float)\n        values[np.abs(values) < 1e-4] = 0  # Only round declared solver-scale deviations.\n        ax.bar(np.arange(3)+(index-1)*.23, values, width=.22, color=COUNTRY_COLORS[country])\n    ax.set_xticks([0, 1, 2], ["Early", "Late", "No change"])\n    ax.set_title("C) Mean simulated peak reduction", fontsize=10, fontweight="bold")\n    ax.set_ylabel("Peak reduction (%)", fontsize=9)\n    _axis_style(ax)\n    differences = np.vstack([data["policy_waves"].loc[lambda x: x.Country.eq(c)].sort_values("Wave").eval("Early_Reduction_Percent - Late_Reduction_Percent").to_numpy() for c in COUNTRY_ORDER])\n    ax = axes[1, 1]\n    lower = min(0, differences.min())\n    mesh = ax.pcolormesh(differences, cmap="Blues" if lower == 0 else "RdBu_r",\n        vmin=lower, vmax=max(1, differences.max()), shading="flat")\n    ax.invert_yaxis()\n    ax.set_xticks(np.arange(3)+.5, ["Wave 1", "Wave 2", "Wave 3"])\n    ax.set_yticks(np.arange(3)+.5, [country_display(c) for c in COUNTRY_ORDER], rotation=0)\n    for r in range(3):\n        for c in range(3):\n            ax.text(c+.5, r+.5, f"{differences[r,c]:.1f}", ha="center", va="center", fontsize=9,\n                color="white" if differences[r,c] > differences.max()*.6 else "black")\n    ax.set_title("D) Early minus late by wave", fontsize=10, fontweight="bold")\n    bar = fig.colorbar(mesh, ax=ax, fraction=.055, pad=.03)\n    bar.set_label("Percentage points", fontsize=9)\n    return fig\n\n\ndef _parameter_series(parameter_wide, country, wave, parameter):\n    frame = parameter_wide.loc[parameter_wide.Country.eq(country) & parameter_wide.Wave.eq(f"Wave {wave}") & parameter_wide.Parameter.eq(parameter)].sort_values("Date")\n    if frame.empty:\n        raise ValueError(f"No parameter estimates for {country}, Wave {wave}, {parameter}.")\n    return frame\n\n\ndef plot_parameter_overview(parameter_wide):\n    fig, axes = plt.subplots(3, 2, figsize=(7, 6.6))\n    for row, country in enumerate(COUNTRY_ORDER):\n        for col, parameter in enumerate(("beta_f", "mu_f")):\n            ax = axes[row, col]\n            for wave in (1, 2, 3):\n                frame = _parameter_series(parameter_wide, country, wave, parameter)\n                ax.axvspan(frame.Date.iloc[0], frame.Date.iloc[-1], color="#e8eef4" if wave % 2 else "#f4f4f4")\n                ax.plot(frame.Date, frame.EnKPF, lw=1.25, color=METHOD_COLORS["EnKF"])\n                if wave > 1: ax.axvline(frame.Date.iloc[0], color=".5", ls="--", lw=.7)\n            ax.set_title(f"{country_display(country)}: {PARAMETER_LABELS[parameter]}", fontsize=10, fontweight="bold")\n            ax.set_ylabel(PARAMETER_LABELS[parameter], fontsize=9)\n            if row == 2: ax.set_xlabel("Reporting date", fontsize=9)\n            _axis_style(ax); _date_style(ax)\n    return fig\n\n\ndef behavioral_change_table(parameter_wide):\n    rows = []\n    for country in COUNTRY_ORDER:\n        for wave in (1, 2, 3):\n            for parameter in ("beta_f", "mu_f"):\n                frame = _parameter_series(parameter_wide, country, wave, parameter)\n                start, end = float(frame.EnKPF.iloc[0]), float(frame.EnKPF.iloc[-1])\n                rows.append({"Country": country, "Wave": wave, "Parameter": parameter,\n                    "Start_date": frame.Date.iloc[0], "End_date": frame.Date.iloc[-1], "Start": start,\n                    "End": end, "Mean": frame.EnKPF.mean(),\n                    "Percent_change": (end-start)/abs(start)*100 if start != 0 else np.nan})\n    return pd.DataFrame(rows)\n\n\ndef plot_within_wave_changes(parameter_wide):\n    fig, axes = plt.subplots(9, 2, figsize=(7, 9.3))\n    for ci, country in enumerate(COUNTRY_ORDER):\n        for wi, wave in enumerate((1, 2, 3)):\n            for col, parameter in enumerate(("beta_f", "mu_f")):\n                ax = axes[ci*3+wi, col]\n                frame = _parameter_series(parameter_wide, country, wave, parameter)\n                values = frame.EnKPF.to_numpy(float)\n                ax.plot(frame.Date, values, lw=1.3, color="#0072B2")\n                change = (values[-1]-values[0])/abs(values[0])*100 if values[0] != 0 else np.nan\n                if np.isfinite(change):\n                    ax.text(.98, .9, ("↑" if change >= 0 else "↓")+f"{abs(change):.1f}%",\n                        transform=ax.transAxes, ha="right", va="top", fontsize=10.5,\n                        color="#d62728" if change >= 0 else "#228b22", fontweight="bold",\n                        bbox={"facecolor":"white", "edgecolor":".75", "boxstyle":"round,pad=.15"})\n                ax.set_title(f"{country_display(country)} — Wave {wave}", fontsize=10.5, fontweight="bold", pad=3)\n                ax.set_xlim(frame.Date.iloc[0], frame.Date.iloc[-1])\n                _axis_style(ax)\n    return fig\n\n\ndef plot_parameter_comparison(parameter_wide, country):\n    fig, axes = plt.subplots(3, 4, figsize=(7, 7.8))\n    for row, wave in enumerate((1, 2, 3)):\n        for col, parameter in enumerate(PARAMETERS):\n            ax = axes[row, col]\n            frame = _parameter_series(parameter_wide, country, wave, parameter)\n            for method in METHOD_COLORS:\n                ax.plot(frame.Date, frame[method], color=METHOD_COLORS[method], lw=1.1,\n                    ls="--" if method == "EnKPF" else "-", label=method)\n            ax.set_title(f"Wave {wave}: {PARAMETER_LABELS[parameter]}", fontsize=10, fontweight="bold")\n            ax.set_xlabel("Date", fontsize=9)\n            if col == 0: ax.set_ylabel("Estimate", fontsize=9)\n            ax.set_xlim(frame.Date.iloc[0], frame.Date.iloc[-1])\n            ax.xaxis.set_major_locator(FixedLocator(ax.get_xlim()))\n            ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))\n            ax.tick_params(axis="x", labelrotation=20)\n            _axis_style(ax)\n    return fig\n\n\ndef output_index(tables, table_paths, png_dir, tiff_dir):\n    rows = []\n    for section, number, title, stem in FIGURES:\n        png, tif, pdf = Path(png_dir)/f"{stem}.png", Path(tiff_dir)/f"{stem}.tif", Path(png_dir)/f"{stem}.pdf"\n        for path in (png, tif, pdf):\n            if not path.is_file(): raise FileNotFoundError(f"Run the figure cells first: {path}")\n        rows.append({"Section":section, "Reference":f"Figure {number}", "Title":title,\n            "PNG":str(png), "TIFF":str(tif), "PDF":str(pdf), "CSV":"", "TeX":""})\n    for stem, item in tables.items():\n        info = table_paths["tables"][stem]\n        rows.append({"Section":"Main" if stem.startswith("Main") else "Supplementary",\n            "Reference": "Table " + stem.rsplit("_",1)[1], "Title":item["caption"],\n            "PNG":"", "TIFF":"", "PDF":"", "CSV":str(info["csv"]), "TeX":str(info["tex"])})\n    return pd.DataFrame(rows)\n\n\ndef write_run_manifest(path, project_root, data, figures_and_tables, jb, mardia, forecasts, sensitivity, settings, table_metadata=None):\n    path = Path(path)\n    def relative(value: object) -> str:\n        if isinstance(value, (str, Path)):\n            if isinstance(value, str) and not value.strip():\n                return ""\n            return Path(value).relative_to(project_root).as_posix()\n        if value is None or value is pd.NA:\n            return ""\n        if isinstance(value, float) and np.isnan(value):\n            return ""\n        raise TypeError(f"Expected an output path or a missing value, got {type(value).__name__}.")\n    outputs = figures_and_tables.copy()\n    for column in ("PNG", "TIFF", "PDF", "CSV", "TeX"):\n        outputs[column] = outputs[column].map(relative)\n    source_records = {row["Source"]: row for row in data["sources"].to_dict("records")}\n    if table_metadata is not None:\n        for item in table_metadata.values():\n            for value in item.get("sources", []):\n                source = Path(value)\n                source = source if source.is_absolute() else Path(project_root) / source\n                if source.is_file():\n                    row = _source_record(source, project_root)\n                    source_records[row["Source"]] = row\n    protocol = policy_protocol(data["policy_waves"])\n    manifest = {"Created_UTC":datetime.now(timezone.utc).isoformat(), "Settings":settings,\n        "Policy_protocol": protocol,\n        "Sources":list(source_records.values()), "Forecast_sources":forecasts.sources.to_dict("records"),\n        "Outputs":outputs.to_dict("records"), "Gaussianity":{"JB_rejections":int(jb.JB_reject.sum()),\n            "JB_marginals":len(jb), "Mardia_rejections":int(mardia.Reject_Joint.sum()), "Mardia_ensembles":len(mardia)},\n        "Sensitivity_diagnostics":sensitivity["diagnostics"].to_dict("records"),\n        "Interpretation":["Full-fit bands are mean ±1.96 ensemble SD.",\n            "Forecast bands are saved empirical ensemble percentiles, not observation-noise prediction intervals.",\n            "Peak origins are retrospective; Korea uses the second small peak within Wave 1.",\n            protocol["description"] + " " + protocol["interpretation"]]}\n    path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")\n    return path\n', 'manuscript_table_exports': 'from collections import OrderedDict\nfrom pathlib import Path\nimport math\nimport numpy as np\nimport pandas as pd\nfrom openpyxl.styles import Alignment, Font, PatternFill\nfrom openpyxl.utils import get_column_letter\n\ndef _escape_tex(value):\n    replacements = {"\\\\": r"\\textbackslash{}", "&": r"\\&", "%": r"\\%", "$": r"\\$", "#": r"\\#", "_": r"\\_", "{": r"\\{", "}": r"\\}", "~": r"\\textasciitilde{}", "^": r"\\textasciicircum{}"}\n    return "".join(replacements.get(char, char) for char in str(value))\n\n\ndef _tex_number(value):\n    if not np.isfinite(value):\n        raise ValueError("Cannot export a nonfinite manuscript-table number.")\n    if value == 0:\n        return "0"\n    if abs(value) < .001 or abs(value) >= 1e6:\n        return f"{value:.3g}"\n    if float(value).is_integer():\n        return f"{value:,.0f}"\n    return f"{value:,.4f}".rstrip("0").rstrip(".")\n\n\ndef export_publication_tables(tables, tables_dir, tex_dir):\n    """Write one styled workbook plus a CSV and TeX fragment per table.\n\n    The workbook includes an Index sheet with exact captions/labels/notes and\n    source paths. TeX needs booktabs, longtable and array in the parent document.\n    """\n    tables_dir, tex_dir = Path(tables_dir), Path(tex_dir)\n    tables_dir.mkdir(parents=True, exist_ok=True); tex_dir.mkdir(parents=True, exist_ok=True)\n    workbook_path = tables_dir / "Manuscript_Supplementary_Tables.xlsx"\n    index_rows, paths = [], OrderedDict()\n    for stem, item in tables.items():\n        if not {"data", "caption", "label", "notes"}.issubset(item):\n            raise ValueError(f"{stem}: incomplete publication-table metadata.")\n        frame = item["data"]\n        if not isinstance(frame, pd.DataFrame) or frame.empty:\n            raise ValueError(f"{stem}: an export requires a nonempty DataFrame.")\n        csv_path, tex_path = tables_dir / f"{stem}.csv", tex_dir / f"{stem}.tex"\n        frame.to_csv(csv_path, index=False, float_format="%.15g")\n        is_protocol = stem == "Supplementary_Table_S14"\n        if is_protocol:\n            column_format = r">{\\raggedright\\arraybackslash}p{0.18\\linewidth}>{\\raggedright\\arraybackslash}p{0.34\\linewidth}>{\\raggedright\\arraybackslash}p{0.36\\linewidth}"\n        elif len(frame.columns) >= 5:\n            # Fixed-width paragraph columns wrap descriptive headings and state\n            # rules. Their total leaves room for intercolumn padding on A4.\n            widths = [.17] + [.66 / (len(frame.columns) - 1)] * (len(frame.columns) - 1)\n            column_format = "".join(\n                (r">{\\raggedleft\\arraybackslash}" if pd.api.types.is_numeric_dtype(frame[col]) else r">{\\raggedright\\arraybackslash}")\n                + f"p{{{width:.4f}\\\\linewidth}}" for col, width in zip(frame.columns, widths)\n            )\n        else:\n            column_format = "".join("r" if pd.api.types.is_numeric_dtype(frame[col]) else "l" for col in frame.columns)\n        tex = frame.to_latex(index=False, escape=True, longtable=True, float_format=_tex_number,\n                            caption=_escape_tex(item["caption"]), label=item["label"], column_format=column_format)\n        # A local group controls only typography, never the document\'s numbering.\n        note = _escape_tex(item["notes"])\n        tex = "% Generated from current MATLAB exports. Include in the correct main/SI sequence.\\n" + r"\\begingroup\\small\\setlength{\\tabcolsep}{3pt}" + "\\n" + tex + r"\\par\\noindent\\footnotesize\\textit{Note.} " + note + "\\n" + r"\\endgroup" + "\\n"\n        tex_path.write_text(tex, encoding="utf-8")\n        section, number = ("Main", stem.rsplit("_", 1)[1]) if stem.startswith("Main") else ("Supplementary", stem.rsplit("_", 1)[1])\n        index_rows.append({"Output": stem, "Reference": f"Table {number}", "Section": section,\n                           "Caption": item["caption"], "Label": item["label"], "Notes": item["notes"],\n                           "Sources": "; ".join(item.get("sources", [])), "CSV": csv_path.relative_to(tables_dir.parent.parent).as_posix(), "TeX": tex_path.relative_to(tables_dir.parent.parent).as_posix(), "Rows": len(frame)})\n        paths[stem] = {"csv": csv_path, "tex": tex_path, "sheet": stem}\n    index = pd.DataFrame(index_rows)\n    index_path = tables_dir / "Manuscript_Supplementary_Tables_Index.csv"\n    index.to_csv(index_path, index=False)\n    with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:\n        index.to_excel(writer, sheet_name="Index", index=False)\n        for stem, item in tables.items():\n            item["data"].to_excel(writer, sheet_name=stem, index=False)\n        for sheet in writer.book.worksheets:\n            sheet.freeze_panes = "A2"\n            sheet.auto_filter.ref = sheet.dimensions\n            for cell in sheet[1]:\n                cell.font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")\n                cell.fill = PatternFill("solid", fgColor="234E70")\n                cell.alignment = Alignment(wrap_text=True, vertical="center")\n            sheet.row_dimensions[1].height = 32\n            for row in sheet.iter_rows(min_row=2):\n                for cell in row:\n                    cell.font = Font(name="Calibri", size=11)\n                    cell.alignment = Alignment(vertical="top", wrap_text=isinstance(cell.value, str))\n                    if cell.row % 2 == 0:\n                        cell.fill = PatternFill("solid", fgColor="F0F5F9")\n                    if isinstance(cell.value, (int, float)):\n                        heading = str(sheet.cell(1, cell.column).value or "")\n                        integer_columns = {"Wave", "Wave 1", "Wave 2", "Wave 3", "N0", "S0", "I0", "Q0", "R0", "I0 start", "Seed", "Rows", "Train_Days", "Test_Days", "Duration (days)", "Min parents"}\n                        if heading in integer_columns and float(cell.value).is_integer():\n                            cell.number_format = "#,##0;[Red]-#,##0;0"\n                        elif 0 < abs(float(cell.value)) < .001:\n                            cell.number_format = "0.000E+00"\n                        else:\n                            cell.number_format = "#,##0.0000;[Red]-#,##0.0000;0"\n                    elif sheet.title == "Supplementary_Table_S16" and sheet.cell(1, cell.column).value in {"Start", "End"}:\n                        cell.number_format = "yyyy-mm-dd"\n            for col_num, column in enumerate(sheet.columns, 1):\n                content = max(len(str(cell.value or "")) for cell in column)\n                sheet.column_dimensions[get_column_letter(col_num)].width = min(55, max(13, content + 2))\n            for row in sheet.iter_rows(min_row=2):\n                wrapped_lines = 1\n                for cell in row:\n                    if isinstance(cell.value, str):\n                        width = sheet.column_dimensions[get_column_letter(cell.column)].width\n                        # Excel\'s Calibri width units approximate character widths.\n                        # Leave a small margin for proportional lettering and padding.\n                        usable_width = max(width - 3, 8)\n                        lines = sum(max(1, math.ceil(len(line) / usable_width))\n                                    for line in cell.value.splitlines() or [""])\n                        wrapped_lines = max(wrapped_lines, lines)\n                sheet.row_dimensions[row[0].row].height = min(150, max(20, 15 * wrapped_lines + 4))\n            sheet.sheet_properties.pageSetUpPr.fitToPage = True\n            sheet.page_setup.orientation = "landscape" if sheet.max_column > 5 else "portrait"\n            sheet.page_setup.paperSize = sheet.PAPERSIZE_A4\n            sheet.page_setup.fitToWidth = 1\n            sheet.page_setup.fitToHeight = 0\n            sheet.print_title_rows = "1:1"\n    return {"workbook": workbook_path, "index": index_path, "tables": paths}\n', 'publication_diagnostics': '"""Endpoint Gaussianity diagnostics from the current hybrid-run EnKF ensembles."""\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nfrom matplotlib.lines import Line2D\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nfrom scipy.interpolate import PchipInterpolator\nfrom scipy.io import loadmat\nfrom scipy.optimize import brentq\nfrom scipy.stats import chi2, kurtosis, norm, skew\nfrom publication_styles import country_display\n\n\nCOUNTRY_WORKSPACES = {\n    "South Korea": "Korea_EnKPF_G_workspace.mat",\n    "Japan": "Jap_EnKPF_G_workspace.mat",\n    "Pakistan": "Pak_EnKPF_G_workspace.mat",\n}\nPARAMETERS = ("beta", "beta_f", "mu_f", "I0")\nPARAMETER_LABELS = (r"$\\beta$", r"$\\beta_f$", r"$\\mu_f$", r"$I_0$")\nCOLORS = ("#0072B2", "#D55E00", "#009E73", "#CC79A7")\n# MATLAB R2026a jbtest\'s finite-sample critical-value row for n=200.\n# These numerical critical values are used with its PCHIP alpha interpolation.\nJB_ALPHAS = np.array([.001, .0016681, .0027826, .0046416, .0077426, .01,\n    .012915, .021544, .025, .035938, .05, .059948, .1, .15, .2, .25, .5])\nJB_CRITICAL_200 = np.array([27.5140, 23.1914, 19.3922, 16.0635, 13.1643,\n    11.8602, 10.6494, 8.4904, 7.9259, 6.6688, 5.6783, 5.1957,\n    4.0327, 3.2584, 2.7559, 2.3812, 1.2382])\n\n\ndef jb_matlab_200(values):\n    """Return JB statistic, tabulated finite-sample p, and rejection at .05."""\n    values = np.asarray(values, dtype=float)\n    if values.shape != (200,) or not np.isfinite(values).all() or np.var(values) == 0:\n        raise ValueError("Finite-sample JB diagnostics require 200 finite, nonconstant members.")\n    statistic = 200 * (skew(values, bias=True)**2 / 6\n        + (kurtosis(values, fisher=False, bias=True) - 3)**2 / 24)\n    critical = PchipInterpolator(JB_ALPHAS, JB_CRITICAL_200)\n    if statistic >= JB_CRITICAL_200[0]:\n        p_value, bound = float(JB_ALPHAS[0]), "at most"\n    elif statistic <= JB_CRITICAL_200[-1]:\n        p_value, bound = float(JB_ALPHAS[-1]), "at least"\n    else:\n        p_value = brentq(lambda alpha: float(critical(alpha)) - statistic,\n            float(JB_ALPHAS[0]), float(JB_ALPHAS[-1]))\n        bound = "interpolated"\n    return float(statistic), p_value, bool(statistic > critical(.05)), bound\n\n\ndef mardia_matlab(values):\n    """Match Functions_new/mardiaTest.m (sample covariance and pinv)."""\n    values = np.asarray(values, dtype=float)\n    n, p = values.shape\n    centered = values - values.mean(axis=0)\n    covariance = np.cov(values, rowvar=False, ddof=1)\n    scale = np.linalg.norm(covariance, 2)\n    if not np.isfinite(scale) or scale <= 0:\n        raise ValueError("Mardia diagnostics require a finite, nonzero covariance.")\n    tolerance = max(covariance.shape) * np.spacing(scale)\n    inverse = np.linalg.pinv(covariance, rcond=tolerance / scale)\n    distances = centered @ inverse @ centered.T\n    b1 = np.sum(distances**3) / n**2\n    b2 = np.mean(np.diag(distances)**2)\n    skew_statistic = n * b1 / 6\n    kurt_statistic = (b2 - p * (p + 2)) / np.sqrt(8 * p * (p + 2) / n)\n    p_skew = float(1 - chi2.cdf(skew_statistic, p * (p + 1) * (p + 2) / 6))\n    p_kurt = float(2 * (1 - norm.cdf(abs(kurt_statistic))))\n    return {"Mardia_Skewness": b1, "Mardia_Kurtosis": b2,\n        "Skewness_Statistic": skew_statistic, "Kurtosis_Statistic": kurt_statistic,\n        "p_skew": p_skew, "p_kurt": p_kurt,\n        "Reject_Joint": p_skew < .05 or p_kurt < .05,\n        "Covariance_Rank": int(np.linalg.matrix_rank(covariance, tol=tolerance))}\n\n\ndef current_endpoint_diagnostics(project_root, countries):\n    marginal_rows, joint_rows = [], []\n    for country, filename in COUNTRY_WORKSPACES.items():\n        path = Path(countries[country]) / "Results" / "Results_EnKPF_LSM" / filename\n        keys = [f"ens_his{wave}" for wave in (1, 2, 3)]\n        data = loadmat(path, variable_names=keys)\n        for wave in (1, 2, 3):\n            key = f"ens_his{wave}"\n            if key not in data:\n                raise KeyError(f"{path}: current EnKF ensemble {key} is missing.")\n            history = np.asarray(data[key], dtype=float)\n            if history.ndim != 3 or history.shape[:2] != (9, 200):\n                raise ValueError(f"{path}: expected 9 x 200 x day {key}, got {history.shape}.")\n            values = history[5:9, :, -1].T\n            if not np.isfinite(values).all():\n                raise ValueError(f"{country} Wave {wave}: nonfinite endpoint parameter ensemble.")\n            provenance = {"Country": country, "Wave": wave, "N_ensemble": len(values),\n                "Source": str(path.relative_to(project_root)), "Ensemble": key,\n                "Time_Summary": "Final analysis ensemble of this wave"}\n            for index, parameter in enumerate(PARAMETERS):\n                column = values[:, index]\n                statistic, p_value, reject, bound = jb_matlab_200(column)\n                marginal_rows.append({**provenance, "Parameter": parameter,\n                    "JB_Statistic": statistic, "JB_p_value": p_value,\n                    "JB_p_bound": bound, "JB_reject": reject,\n                    "Skewness": float(skew(column, bias=True)),\n                    "Kurtosis": float(kurtosis(column, fisher=False, bias=True))})\n            joint_rows.append({**provenance, **mardia_matlab(values)})\n    return pd.DataFrame(marginal_rows), pd.DataFrame(joint_rows)\n\n\ndef _save(fig, stem, png_dir, tiff_dir):\n    png_dir, tiff_dir = Path(png_dir), Path(tiff_dir)\n    png_dir.mkdir(parents=True, exist_ok=True)\n    tiff_dir.mkdir(parents=True, exist_ok=True)\n    png = png_dir / f"{stem}.png"\n    fig.savefig(png, dpi=600, facecolor="white", bbox_inches="tight")\n    fig.savefig(png_dir / f"{stem}.pdf", facecolor="white", bbox_inches="tight")\n    with Image.open(png) as raster:\n        raster.convert("RGB").save(tiff_dir / f"{stem}.tif",\n            compression="tiff_lzw", dpi=(600, 600))\n\n\ndef _style(ax):\n    ax.tick_params(labelsize=9)\n    ax.set_xticks([1, 2, 3], ["1", "2", "3"])\n    ax.set_xlabel("Wave", fontsize=9)\n    ax.grid(axis="y", color="#dddddd", linewidth=.5)\n    ax.set_axisbelow(True)\n    for side in ("top", "right"):\n        ax.spines[side].set_visible(False)\n\n\ndef refresh_gaussianity_figures(project_root, countries, tables_dir,\n        manuscript_tables_dir, png_dir, tiff_dir):\n    """Export matched current endpoint tables and three supplementary figures."""\n    project_root = Path(project_root)\n    marginal, joint = current_endpoint_diagnostics(project_root, countries)\n    tables_dir, manuscript_tables_dir = Path(tables_dir), Path(manuscript_tables_dir)\n    tables_dir.mkdir(parents=True, exist_ok=True)\n    manuscript_tables_dir.mkdir(parents=True, exist_ok=True)\n    marginal.to_csv(tables_dir / "current_hybrid_EnKF_JB_endpoints.csv", index=False)\n    joint.to_csv(tables_dir / "current_hybrid_EnKF_Mardia_endpoints.csv", index=False)\n    # Full precision remains in CSV; manuscript p-value bounds are explicit.\n    shown = marginal[["Country", "Wave", "Parameter", "JB_Statistic", "JB_p_value",\n        "JB_p_bound", "JB_reject", "Skewness", "Kurtosis"]].copy()\n    shown.to_latex(manuscript_tables_dir / "current_hybrid_EnKF_JB_endpoints.tex", index=False,\n        float_format="%.5g", escape=True)\n    joint[["Country", "Wave", "p_skew", "p_kurt", "Reject_Joint"]].to_latex(\n        manuscript_tables_dir / "current_hybrid_EnKF_Mardia_endpoints.tex", index=False,\n        float_format="%.5g", escape=True)\n    country_order = list(COUNTRY_WORKSPACES)\n    handles = [Line2D([0], [0], color=color, lw=3, label=label)\n        for color, label in zip(COLORS, PARAMETER_LABELS)]\n    with plt.rc_context({"font.family": "sans-serif", "font.size": 9}):\n        fig, axes = plt.subplots(2, 3, figsize=(7, 5.7), constrained_layout=True)\n        for col, country in enumerate(country_order):\n            rows = marginal[marginal.Country.eq(country)]\n            ax = axes[0, col]\n            for index, parameter in enumerate(PARAMETERS):\n                subset = rows[rows.Parameter.eq(parameter)].sort_values("Wave")\n                x = subset.Wave.to_numpy() + (index-1.5)*.16\n                values = subset.JB_p_value.to_numpy()\n                ax.bar(x, np.maximum(values, 1e-4), width=.15, color=COLORS[index])\n            ax.set_title(country_display(country), fontsize=10, fontweight="bold")\n            ax.set_yscale("log")\n            ax.set_ylim(1e-4, 1)\n            ax.axhline(.05, color="black", ls="--", lw=.8)\n            _style(ax)\n            if col == 0:\n                ax.set_ylabel("Marginal JB p-value", fontsize=9)\n            subset = joint[joint.Country.eq(country)].sort_values("Wave")\n            ax = axes[1, col]\n            for key, marker, color in [("p_skew", "o", "#0072B2"), ("p_kurt", "s", "#D55E00")]:\n                values = subset[key].to_numpy()\n                ax.plot(subset.Wave, np.maximum(values, 1e-4), marker=marker,\n                    color=color, lw=1, ms=4)\n                below = values < 1e-4\n                if below.any():\n                    ax.scatter(subset.Wave.to_numpy()[below], np.full(below.sum(), 1.2e-4),\n                        marker="v", color=color, s=25, zorder=5)\n            ax.set_yscale("log")\n            ax.set_ylim(1e-4, 1)\n            ax.axhline(.05, color="black", ls="--", lw=.8)\n            _style(ax)\n            if col == 0:\n                ax.set_ylabel("Mardia p-value", fontsize=9)\n        fig.legend(handles=handles, loc="outside upper center", ncol=4, frameon=False, fontsize=9)\n        fig.legend(handles=[Line2D([0],[0],color="#0072B2",marker="o",label="Joint skewness"),\n            Line2D([0],[0],color="#D55E00",marker="s",label="Joint kurtosis")],\n            loc="outside lower center",ncol=2,frameon=False,fontsize=9)\n        _save(fig, "Supplementary_Figure_1_p_values", png_dir, tiff_dir)\n        plt.close(fig)\n        for field, stem, ylabel, offset in [\n            ("Skewness", "Supplementary_Figure_2_Skewness", "Marginal skewness", 0),\n            ("Kurtosis", "Supplementary_Figure_3_Kurtosis", "Marginal excess kurtosis", 3)]:\n            fig, axes = plt.subplots(1, 3, figsize=(7, 2.8), constrained_layout=True)\n            for col, country in enumerate(country_order):\n                ax = axes[col]\n                for index, parameter in enumerate(PARAMETERS):\n                    subset = marginal[marginal.Country.eq(country) & marginal.Parameter.eq(parameter)].sort_values("Wave")\n                    ax.bar(subset.Wave.to_numpy()+(index-1.5)*.16,\n                        subset[field].to_numpy()-offset,width=.15,color=COLORS[index])\n                ax.axhline(0,color="black",lw=.7)\n                ax.set_title(country_display(country),fontsize=10,fontweight="bold")\n                _style(ax)\n                if col == 0: ax.set_ylabel(ylabel,fontsize=9)\n            fig.legend(handles=handles,loc="outside upper center",ncol=4,frameon=False,fontsize=9)\n            _save(fig,stem,png_dir,tiff_dir)\n            plt.close(fig)\n    print(f"Current hybrid-run EnKF endpoints: JB rejects {marginal.JB_reject.sum()}/36; "\n        f"Mardia rejects {joint.Reject_Joint.sum()}/9 (alpha=0.05).")\n    print("JB p-values use the n=200 finite-sample table; Mardia values below the plotting floor are marked.")\n    return marginal, joint\n', 'amm_publication_outputs': 'from collections import OrderedDict\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\ndef cached_table_paths(project_root, tables):\n    """Resolve source paths for a read-only review without rewriting workbooks."""\n    root = Path(project_root).resolve()\n    paths = OrderedDict()\n    for key, item in tables.items():\n        csv = root / item.get("csv_source", f"Results/Tables/{key}.csv")\n        tex = root / item["tex_source"] if item.get("tex_source") else ""\n        paths[key] = {"csv": csv, "tex": tex, "sheet": key}\n    return {"workbook": root / "Results/Tables/Manuscript_Supplementary_Tables.xlsx", "tables": paths}\n\n\ndef style_publication_table(item):\n    """Use manuscript rounding for display while retaining original numeric data."""\n    frame = item.get("display_data", item["data"])\n    label = item["label"]\n    if label in {"tab:amm-recovery", "tab:amm-information"}:\n        specification = ".3g"\n    elif label in {"tab:amm-primary-validation", "tab:amm-window-validation",\n                   "tab:amm-horizons", "tab:amm-fixed-sensitivity",\n                   "tab:amm-paired-contrasts", "tab:amm-legacy-operator"}:\n        specification = ".3f"\n    else:\n        specification = ".5g"\n\n    def format_value(value):\n        if isinstance(value, (int, np.integer, bool, np.bool_)):\n            return str(int(value))\n        if isinstance(value, (float, np.floating)):\n            return format(value, specification)\n        return value\n\n    return frame.style.format(format_value).hide(axis="index")\n', 'focused_peak_outputs': '"""Publication outputs for all completed native peak-forecast experiments.\n\nReads cached native MATLAB CSV files only; no fitting, simulation, sampling,\nforecast repair, or outcome-based seed selection is performed here. The\nAll completed country protocols are retained, including Pakistan\'s budget-limited\nLSM initializer. A separate positive-exit-flag subset preserves the retrospective\nconvergence check without selecting countries or seeds by forecast error.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nfrom datetime import datetime, timedelta\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nfrom statistics import mean, stdev\n\n\nROOT = Path(PACKAGE_ROOT)\nRESULTS = ROOT / "Data/NativeForecast"\nOUTPUTS = ROOT / "Results/Forecast"\nPROTOCOL = "NativeMATLAB_OriginalFilters_20261020_v1"\nIMPLEMENTATION = "1ab575d44c3fc57f9e71d98784b837d4fce43442fd91342cc12e1d254b0ba421"\nCOUNTRIES = ("Korea", "Japan", "Pakistan")\nMETHODS = ("EnKF", "EnKPF", "StaticLSM", "Persistence")\nSEEDS = (0, 1, 123, 2026, 202601)\nHORIZONS = ("14-day", "Full remainder")\nFIGURE_STEM = "Focused_EnKPF_Peak_Forecasts"\nELIGIBILITY_RULE = (\n    "Retain all three completed original country-specific peak protocols and every "\n    "declared computational seed for every method. Report the training optimizer "\n    "termination flag separately; do not select countries or seeds by forecast error."\n)\nELIGIBILITY_CHRONOLOGY = (\n    "The earlier retrospective positive-exit-flag restriction is retained as a "\n    "converged-initializer sensitivity subset. The completed Pakistani experiment "\n    "is restored to the primary descriptive analysis with its evaluation-limit "\n    "termination disclosed. Neither analysis is preregistered or independent validation."\n)\nCONVERGENCE_RULE = (\n    "For the sensitivity subset, retain an entire country-specific origin only if "\n    "the original LSM initializer and all five declared repeated-seed initializer "\n    "records have positive OptimizerExitFlag values. Apply the origin-level rule "\n    "uniformly to all methods and outcomes, independently of forecast scores."\n)\n\n\ndef _read(path: Path) -> list[dict[str, str]]:\n    with path.open(encoding="utf-8-sig", newline="") as stream:\n        reader = csv.DictReader(stream)\n        assert reader.fieldnames and len(reader.fieldnames) == len(set(reader.fieldnames))\n        rows = list(reader)\n    assert rows, f"Empty source: {path}"\n    return rows\n\n\ndef _num(row: dict, key: str) -> float:\n    value = float(row[key])\n    assert math.isfinite(value), f"Nonfinite {key}: {row.get(\'RunID\', row)}"\n    return value\n\n\ndef _close(actual: float, expected: float) -> None:\n    assert math.isclose(actual, expected, rel_tol=1e-9, abs_tol=1e-6), (actual, expected)\n\n\ndef _sha(path: Path) -> str:\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef _write_csv(path: Path, rows: list[dict]) -> None:\n    assert rows\n    with path.open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef _summarize_repetitions(metrics: list[dict], countries: tuple[str, ...]) -> tuple[list[dict], dict]:\n    """Equal-weight all declared seeds/outcomes at every selected country origin."""\n    summaries = []\n    comparisons = {}\n    paired_outcomes = len(countries) * len(SEEDS) * 2\n    for horizon in HORIZONS:\n        selected = [r for r in metrics if r["Country"] in countries\n                    and r["Period"] == "Test" and r["HorizonLabel"] == horizon]\n        assert len(selected) == paired_outcomes * len(METHODS)\n        pairs = {(r["Country"], int(r["Seed"]), r["Outcome"], r["Method"]): r for r in selected}\n        assert len(pairs) == len(selected)\n        count = sum(_num(pairs[(country, seed, outcome, "EnKPF")], "RMSE") <\n                    _num(pairs[(country, seed, outcome, "EnKF")], "RMSE")\n                    for country in countries for seed in SEEDS for outcome in ("Q", "R"))\n        comparisons[horizon] = {"EnKPF_lower_RMSE_than_EnKF": count,\n                                "PairedOutcomes": paired_outcomes}\n        for method in METHODS:\n            rows = [r for r in selected if r["Method"] == method]\n            assert len(rows) == paired_outcomes\n            seed_means = [mean(_num(r, "NRMSE") for r in rows if int(r["Seed"]) == seed)\n                          for seed in SEEDS]\n            summaries.append({\n                "Horizon": horizon, "Method": method, "MacroNRMSE": mean(seed_means),\n                "SharedSeedMacroSD": stdev(seed_means),\n                "LatentCoverage95Percent": mean(_num(r, "LatentCoverage95") for r in rows)\n                if method in ("EnKF", "EnKPF") else None,\n                "Countries": ";".join(countries), "Origins": len(countries), "Seeds": len(SEEDS),\n                "CountrySeedRuns": len(countries) * len(SEEDS), "OutcomeRows": len(rows),\n                "Aggregation": f"Equal Q/R, five seeds, and {len(countries)} country-specific origins",\n            })\n    return summaries, comparisons\n\n\ndef load_focused_data() -> dict:\n    """Read completed sources; expose primary and convergence-only summaries."""\n    modes = {}\n    source_paths = []\n    for mode, expected in (("script_reproduction", 3), ("peak_stability", 15)):\n        folder = RESULTS / mode\n        source = {}\n        for name in ("Run_Manifest", "Fit_Diagnostics", "Forecast_Metrics", "Forecast_Predictions"):\n            path = folder / f"{name}.csv"\n            source_paths.append(path)\n            source[name] = _read(path)\n        manifest = source["Run_Manifest"]\n        assert len(manifest) == expected\n        ids = {row["RunID"] for row in manifest}\n        assert len(ids) == expected and all(row["Status"] == "success" for row in manifest)\n        assert {row["ProtocolID"] for row in manifest} == {PROTOCOL}\n        assert {row["ImplementationHash"] for row in manifest} == {IMPLEMENTATION}\n        assert {int(row["EnsembleSize"]) for row in manifest} == {200}\n        fits = source["Fit_Diagnostics"]\n        assert len(fits) == expected and {row["RunID"] for row in fits} == ids\n        assert {row["ImplementationHash"] for row in fits} == {IMPLEMENTATION}\n        if mode == "peak_stability":\n            assert {(row["Country"], int(row["Seed"])) for row in manifest} == {\n                (country, seed) for country in COUNTRIES for seed in SEEDS\n            }\n        for name in ("Forecast_Metrics", "Forecast_Predictions"):\n            assert {row["RunID"] for row in source[name]} == ids\n            assert {row["ProtocolID"] for row in source[name]} == {PROTOCOL}\n            assert {row["ImplementationHash"] for row in source[name]} == {IMPLEMENTATION}\n        metrics = source["Forecast_Metrics"]\n        assert len(metrics) == len({(r["RunID"], r["Method"], r["Outcome"], r["Period"], r["HorizonLabel"]) for r in metrics})\n        for horizon in HORIZONS:\n            selected = [r for r in metrics if r["Period"] == "Test" and r["HorizonLabel"] == horizon]\n            assert len(selected) == expected * len(METHODS) * 2\n            assert {r["Method"] for r in selected} == set(METHODS)\n        modes[mode] = source\n\n    original = modes["script_reproduction"]\n    repeated = modes["peak_stability"]\n    eligibility = []\n    for country in COUNTRIES:\n        original_fit = [r for r in original["Fit_Diagnostics"] if r["Country"] == country]\n        repeated_fits = [r for r in repeated["Fit_Diagnostics"] if r["Country"] == country]\n        original_run = [r for r in original["Run_Manifest"] if r["Country"] == country]\n        assert len(original_fit) == len(original_run) == 1 and len(repeated_fits) == 5\n        flag = int(original_fit[0]["OptimizerExitFlag"])\n        repeat_flags = [int(r["OptimizerExitFlag"]) for r in repeated_fits]\n        converged = flag > 0 and all(value > 0 for value in repeat_flags)\n        eligibility.append({\n            "Country": country,\n            "OriginalRunID": original_run[0]["RunID"],\n            "OriginalSeed": int(original_run[0]["Seed"]),\n            "OriginalOptimizerExitFlag": flag,\n            "OriginalOptimizerIterations": int(original_fit[0]["Iterations"]),\n            "OriginalOptimizerEvaluations": int(original_fit[0]["Evaluations"]),\n            "RepeatedRuns": len(repeated_fits),\n            "RepeatedSeeds": ";".join(str(seed) for seed in SEEDS),\n            "RepeatedOptimizerExitFlags": ";".join(str(value) for value in repeat_flags),\n            "Included": True,\n            "PrimaryIncluded": True,\n            "ConvergedSubsetIncluded": converged,\n            "Reason": ("Completed forecast; positive training-optimizer exit flags; all five seeds retained."\n                       if converged else "Completed forecast retained; LSM reached its evaluation budget (flag 0); omitted only from the converged-initializer sensitivity subset."),\n        })\n    included = tuple(row["Country"] for row in eligibility if row["PrimaryIncluded"])\n    converged_countries = tuple(row["Country"] for row in eligibility if row["ConvergedSubsetIncluded"])\n    assert included == COUNTRIES and converged_countries == ("Korea", "Japan")\n    assert [r["OriginalOptimizerExitFlag"] for r in eligibility] == [1, 1, 0]\n\n    original_metrics = [r for r in original["Forecast_Metrics"] if r["Country"] in included]\n    repeated_metrics = [r for r in repeated["Forecast_Metrics"] if r["Country"] in included]\n    original_lookup = {(r["Country"], r["Method"], r["Outcome"]): r for r in original_metrics\n                       if r["Period"] == "Test" and r["HorizonLabel"] == "Full remainder"}\n    review_path = RESULTS / "Native_Result_Review.json"\n    source_paths.append(review_path)\n    review = json.loads(review_path.read_text(encoding="utf-8"))\n    references = [r for r in review["script_reproduction"]["original_script_comparison"] if r["Country"] in included]\n    assert len(references) == 12 and all(r["Matches"] for r in references)\n    for reference in references:\n        metric = original_lookup[(reference["Country"], reference["Method"], reference["Outcome"])]\n        _close(_num(metric, "RMSE"), reference["ReferenceRMSE"])\n        _close(_num(metric, "R2"), reference["R2"])\n\n    summaries, comparisons = _summarize_repetitions(repeated_metrics, included)\n    converged_summaries, converged_comparisons = _summarize_repetitions(repeated_metrics, converged_countries)\n\n    original_rows = []\n    for country in included:\n        for outcome in ("Q", "R"):\n            enkf, hybrid = (original_lookup[(country, method, outcome)] for method in ("EnKF", "EnKPF"))\n            assert _num(hybrid, "RMSE") < _num(enkf, "RMSE")\n            original_rows.append({\n                "Country": country, "Outcome": outcome,\n                "RMSE_EnKF": _num(enkf, "RMSE"), "RMSE_EnKPF": _num(hybrid, "RMSE"),\n                "R2_EnKF": _num(enkf, "R2"), "R2_EnKPF": _num(hybrid, "R2"),\n                "ResidualAIC_EnKF": _num(enkf, "ResidualAIC"),\n                "ResidualAIC_EnKPF": _num(hybrid, "ResidualAIC"),\n                "EnKPF_LatentCoverage95Percent": _num(hybrid, "LatentCoverage95"),\n            })\n\n    predictions = [r for r in original["Forecast_Predictions"] if r["Country"] in included and r["Method"] == "EnKPF"]\n    assert len(predictions) == len({(r["RunID"], r["Outcome"], r["Date"]) for r in predictions})\n    for country in included:\n        run = next(r for r in original["Run_Manifest"] if r["Country"] == country)\n        for outcome in ("Q", "R"):\n            rows = sorted((r for r in predictions if r["Country"] == country and r["Outcome"] == outcome), key=lambda r: r["Date"])\n            assert len(rows) == int(run["TrainDays"]) + int(run["TestDays"])\n            dates = [datetime.fromisoformat(r["Date"]) for r in rows]\n            assert all(b - a == timedelta(days=1) for a, b in zip(dates, dates[1:]))\n            assert [r["Period"] for r in rows] == ["Train"] * int(run["TrainDays"]) + ["Test"] * int(run["TestDays"])\n            for row in rows:\n                assert _num(row, "MemberMin") <= _num(row, "Low95") <= _num(row, "High95") <= _num(row, "MemberMax")\n                assert all(_num(row, key) >= 0 for key in ("Observed", "Mean", "Low95", "High95"))\n\n    key_results = {\n        "ProtocolID": PROTOCOL, "ImplementationHash": IMPLEMENTATION,\n        "EligibilityRule": ELIGIBILITY_RULE, "EligibilityChronology": ELIGIBILITY_CHRONOLOGY,\n        "IncludedCountries": list(included), "ExcludedCountries": [],\n        "ExcludedReason": "None of the three completed native peak protocols is excluded from the primary descriptive summary.",\n        "OriginalRuns": 3, "OriginalRMSEValuesMatched": 12,\n        "OriginalEnKPFWins": 6, "OriginalPairedOutcomes": 6,\n        "RepeatedRuns": 15, "Seeds": list(SEEDS), "Comparisons": comparisons,\n        "OriginalResults": original_rows, "Summaries": summaries,\n        "UncertaintyTarget": "Latent central 95% empirical ensemble quantiles; no future measurement noise.",\n        "FullRemainderDurations": {"Korea": 48, "Japan": 55, "Pakistan": 103},\n        "Scope": "All three completed first-window retrospective peak origins, with Pakistan\'s LSM evaluation-limit flag disclosed; all five seeds retained; no unfinished-extension scores or figures.",\n        "ConvergedSubset": {\n            "Rule": CONVERGENCE_RULE, "IncludedCountries": list(converged_countries),\n            "ExcludedCountries": ["Pakistan"], "OriginalRuns": 2,\n            "OriginalEnKPFWins": 4, "OriginalPairedOutcomes": 4,\n            "RepeatedRuns": 10, "Comparisons": converged_comparisons,\n            "Summaries": converged_summaries,\n        },\n        "Sources": {path.relative_to(ROOT).as_posix(): _sha(path) for path in source_paths},\n    }\n    return {"eligibility": eligibility, "included": included, "original": original_rows,\n            "summary": summaries, "predictions": predictions, "key": key_results,\n            "converged_subset": {"included": converged_countries,\n                                 "original": [r for r in original_rows if r["Country"] in converged_countries],\n                                 "summary": converged_summaries, "comparisons": converged_comparisons}}\n\n\ndef original_table(data: dict | None = None) -> str:\n    """Six completed original country/outcome rows; no test-score selection."""\n    data = load_focused_data() if data is None else data\n    rows = []\n    for row in data["original"]:\n        values = [row["Country"], "$" + row["Outcome"] + "$",\n                  _tex_number(row["RMSE_EnKF"]), _tex_number(row["RMSE_EnKPF"]),\n                  _tex_number(row["R2_EnKF"], 3), _tex_number(row["R2_EnKPF"], 3),\n                  f"{row[\'EnKPF_LatentCoverage95Percent\']:.1f}"]\n        rows.append(" & ".join(values) + r" \\\\")\n    return r"""\\begin{table}[htbp]\n\\centering\n\\caption{Completed original peak-origin forecast comparisons for all three countries. Scores cover the full held-out remainder. Coverage is EnKPF\'s latent central 95\\% interval coverage, in percent.}\n\\label{tab:focused-peak-results}\n\\begingroup\\small\\setlength{\\tabcolsep}{3pt}\n\\begin{tabular}{llrrrrr}\n\\toprule\n& & \\multicolumn{2}{c}{RMSE (counts)} & \\multicolumn{2}{c}{$R^2$} & Coverage \\\\\nCountry & Outcome & EnKF & EnKPF & EnKF & EnKPF & EnKPF (\\%) \\\\\n\\midrule\n""" + "\\n".join(rows) + r"""\n\\bottomrule\n\\end{tabular}\\endgroup\n\\par\\smallskip\\begin{minipage}{0.98\\linewidth}\\footnotesize\nKorea uses the second small peak in its first analysis window (130 training/48 held-out days; seed 1); Japan and Pakistan use their first-window active-case maxima (56/55 days, seed 123; and 108/103 days, seed 1, respectively). All runs use 200 members. Pakistan\'s training LSM stops at its function-evaluation budget (exit flag 0; 1,501 evaluations), whereas Korea and Japan have positive termination flags. Its completed forecasts are retained with this initialization limitation; the converged-initializer sensitivity subset excludes Pakistan uniformly across methods and seeds. Negative $R^2$ means larger squared error than the retrospective test-mean reference.\n\\end{minipage}\n\\end{table}\n"""\n\n\ndef repeated_table(data: dict | None = None) -> str:\n    """Show all completed origins and the convergence subset in one compact table."""\n    data = load_focused_data() if data is None else data\n    rows = []\n    for title, summary in (("All completed origins (Korea, Japan, Pakistan)", data["summary"]),\n                           ("Converged-initializer subset (Korea, Japan)", data["converged_subset"]["summary"])):\n        if rows:\n            rows.append(r"\\midrule")\n        rows.append(r"\\multicolumn{3}{l}{\\textit{" + title + r"}} \\\\")\n        for row in summary:\n            if row["Horizon"] != "14-day":\n                continue\n            interval = row["Method"] in ("EnKF", "EnKPF")\n            score = f"{row[\'MacroNRMSE\']:.3f}" + (r" $\\pm$ " + f"{row[\'SharedSeedMacroSD\']:.3f}" if interval else "")\n            coverage = f"{row[\'LatentCoverage95Percent\']:.1f}" if interval else "--"\n            label = {"StaticLSM": "Static LSM"}.get(row["Method"], row["Method"])\n            rows.append(" & ".join((label, score, coverage)) + r" \\\\")\n    return r"""\\begin{table}[htbp]\n\\centering\n\\caption{Fourteen-day repeated-seed forecasts for all three completed peak origins and the converged-initializer sensitivity subset. Both summaries retain all five declared seeds and both outcomes at each included origin.}\n\\label{tab:focused-seed-summary}\n\\begin{tabular}{lrr}\n\\toprule\nMethod & Macro NRMSE & Latent coverage (\\%) \\\\\n\\midrule\n""" + "\\n".join(rows) + r"""\n\\bottomrule\n\\end{tabular}\n\\par\\smallskip\\begin{minipage}{0.98\\linewidth}\\footnotesize\nEach RMSE is divided by its outcome\'s training RMS, floored at one count. Macro scores equally average $Q/R$, five computational seeds ($0,1,123,2026,202601$), and the included country origins. $\\pm$ is the sample SD of five shared-seed macro means, not epidemiological uncertainty. Deterministic references have no assigned distribution or coverage. Pakistan\'s budget-limited training initializer is retained in the primary all-country summary and omitted only from the sensitivity subset, uniformly across all four methods and seeds. The subset is a retrospective convergence check. No completed seed is removed for poor forecast error. Full-remainder scores and all individual repetitions remain in the machine-readable results.\n\\end{minipage}\n\\end{table}\n"""\n\n\ndef eligibility_table(data: dict | None = None) -> str:\n    """Show completion and initializer flags without treating flag 0 as forecast failure."""\n    data = load_focused_data() if data is None else data\n    rows = []\n    for row in data["eligibility"]:\n        cells = [row["Country"], str(row["OriginalOptimizerExitFlag"]),\n                 "Yes" if row["PrimaryIncluded"] else "No",\n                 "Yes" if row["ConvergedSubsetIncluded"] else "No",\n                 "Completed forecast; positive training-optimizer termination." if row["ConvergedSubsetIncluded"]\n                 else "Completed forecast; LSM evaluation-budget stop (1,501 evaluations)."]\n        rows.append(" & ".join(cells) + r" \\\\")\n    return r"""\\begin{table}[htbp]\n\\centering\n\\caption{Completion and training-initializer diagnostics for the native peak forecasts. All completed origins enter the primary analysis; positive optimizer termination flags define a separate sensitivity subset.}\n\\label{tab:focused-eligibility}\n\\begingroup\\small\\setlength{\\tabcolsep}{3pt}\n\\begin{tabular}{lrrr>{\\raggedright\\arraybackslash}p{6.2cm}}\n\\toprule\nCountry & LSM flag & Primary & Subset & Diagnostic \\\\\n\\midrule\n""" + "\\n".join(rows) + r"""\n\\bottomrule\n\\end{tabular}\\endgroup\n\\par\\smallskip\\begin{minipage}{0.98\\linewidth}\\footnotesize\nThe same origin-specific initializer is used in each of the five computational repetitions. A positive termination flag is an optimizer diagnostic, not proof of a global optimum or identifiability. Flag 0 denotes the Pakistani initializer\'s evaluation-limit termination, not failure of its completed forecast simulation. All three origins are retained in the primary descriptive comparison. The earlier positive-flag restriction is retained as a retrospective sensitivity subset; it is applied to all methods and seeds without selecting by held-out error. These forecast-initializer diagnostics do not concern the separate full-wave fitting or behavioral scenario analyses.\n\\end{minipage}\n\\end{table}\n"""\n\n\ndef _tex_number(value: float, digits: int = 2) -> str:\n    if abs(value) >= 1e5:\n        mantissa, exponent = f"{value:.3e}".split("e")\n        return rf"${float(mantissa):.3f}\\times10^{{{int(exponent)}}}$"\n    return f"{value:,.{digits}f}"\n\n\ndef make_figure(data: dict | None = None, output_dir: Path = OUTPUTS, dpi: int = 600) -> dict[str, Path]:\n    """Plot only the declared original EnKPF realizations; include every band endpoint."""\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.dates as mdates\n    import matplotlib.pyplot as plt\n    from matplotlib.lines import Line2D\n    from matplotlib.patches import Patch\n    from matplotlib.ticker import FixedLocator, MaxNLocator\n    import numpy as np\n    from PIL import Image\n\n    data = load_focused_data() if data is None else data\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    rc = {\n        "font.family": "DejaVu Sans", "font.size": 10.5, "axes.labelsize": 10.5,\n        "axes.titlesize": 10.5, "xtick.labelsize": 10.5, "ytick.labelsize": 10.5,\n        "legend.fontsize": 10.5, "pdf.fonttype": 42, "ps.fonttype": 42,\n        "axes.spines.top": False, "axes.spines.right": False,\n    }\n    color = "#c44b20"\n    with plt.rc_context(rc):\n        height = 2.3 * len(data["included"]) + 1.1\n        fig, axes = plt.subplots(len(data["included"]), 2, figsize=(7, height), squeeze=False)\n        fig.subplots_adjust(left=.14, right=.955, top=.95, bottom=.205, hspace=.53, wspace=.48)\n        limits = []\n        for row, country in enumerate(data["included"]):\n            for col, outcome in enumerate(("Q", "R")):\n                ax = axes[row, col]\n                records = sorted((r for r in data["predictions"] if r["Country"] == country and r["Outcome"] == outcome), key=lambda r: r["Date"])\n                dates = np.array([datetime.fromisoformat(r["Date"]) for r in records])\n                observed = np.array([_num(r, "Observed") for r in records])\n                fitted = np.array([_num(r, "Mean") for r in records])\n                lo = np.array([_num(r, "Low95") for r in records])\n                hi = np.array([_num(r, "High95") for r in records])\n                train = sum(r["Period"] == "Train" for r in records)\n                # Bands are latent empirical quantiles; all endpoints are retained.\n                ax.fill_between(dates[train:], lo[train:], hi[train:], color=color, alpha=.20, linewidth=0, zorder=1)\n                ax.plot(dates[:train], fitted[:train], color=color, lw=1.35, zorder=3)\n                ax.plot(dates[train - 1:], fitted[train - 1:], color=color, lw=1.65, ls=(0, (4, 2.5)), zorder=3)\n                ax.plot(dates, observed, color="#202020", marker=".", lw=0, markersize=2.2, zorder=4)\n                ax.axvline(dates[train - 1], color="#6a6a6a", ls=(0, (3, 3)), lw=.9, zorder=2)\n                values = np.concatenate((observed, fitted, lo[train:], hi[train:]))\n                maximum = float(values.max())\n                ax.set_ylim(0, maximum * 1.055 if maximum > 0 else 1)\n                ax.set_xlim(dates[0], dates[-1])\n                assert ax.get_ylim()[1] >= maximum\n                # Exactly three country-specific dates remain readable at final width.\n                ax.xaxis.set_major_locator(FixedLocator(mdates.date2num([dates[0], dates[len(dates) // 2], dates[-1]])))\n                ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))\n                ax.yaxis.set_major_locator(MaxNLocator(nbins=3, min_n_ticks=2))\n                ax.ticklabel_format(axis="y", style="plain", useOffset=False)\n                ax.set_ylabel("Active cases" if outcome == "Q" else "Recovered cases")\n                if row == len(data["included"]) - 1:\n                    ax.set_xlabel("Reporting date")\n                ax.set_title(f"{chr(65 + row * 2 + col)}  {country}: ${outcome}$", loc="left", fontweight="bold", pad=7)\n                ax.grid(axis="y", color="#d9dee5", linewidth=.65)\n                ax.set_axisbelow(True)\n                limits.append({"Country": country, "Outcome": outcome, "MaximumDisplayedValue": maximum,\n                               "AxisMaximum": ax.get_ylim()[1], "ClippedMeanOrBandEndpoints": 0})\n        handles = [\n            Line2D([], [], color="#202020", marker=".", lw=0, markersize=4, label="Observed"),\n            Line2D([], [], color=color, lw=1.4, label="EnKPF fit"),\n            Line2D([], [], color=color, lw=1.65, ls=(0, (4, 2.5)), label="EnKPF forecast"),\n            Patch(facecolor=color, alpha=.20, edgecolor="none", label="95% ensemble interval"),\n            Line2D([], [], color="#6a6a6a", ls=(0, (3, 3)), lw=.9, label="Forecast origin"),\n        ]\n        fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(.53, .015), ncol=3,\n                   frameon=False, columnspacing=1.4, handlelength=2.2, handletextpad=.55)\n        paths = {"PDF": output_dir / f"{FIGURE_STEM}.pdf", "PNG": output_dir / f"{FIGURE_STEM}.png",\n                 "TIFF": output_dir / f"{FIGURE_STEM}.tif"}\n        fig.savefig(paths["PNG"], dpi=dpi, facecolor="white")\n        fig.savefig(paths["PDF"], facecolor="white", metadata={"Title": "EnKPF peak-origin trajectories for all completed original protocols"})\n        plt.close(fig)\n    with Image.open(paths["PNG"]) as raster:\n        raster.convert("RGB").save(paths["TIFF"], compression="tiff_lzw", dpi=(dpi, dpi))\n    provenance = {\n        "Figure": FIGURE_STEM, "SizeInches": [7, height], "RasterDPI": dpi,\n        "ShownMethod": "EnKPF only; EnKF comparison is numerical in the accompanying table.",\n        "Realizations": {r["Country"]: r["OriginalSeed"] for r in data["eligibility"]},\n        "SelectedByForecastError": False,\n        "MeanOrBandClipping": False, "AxisAudit": limits,\n        "Uncertainty": "Central 95% latent ensemble quantiles; no future measurement error.",\n        "Sources": data["key"]["Sources"], "Files": {path.name: _sha(path) for path in paths.values()},\n    }\n    (output_dir / "Focused_Figure_Provenance.json").write_text(json.dumps(provenance, indent=2) + "\\n", encoding="utf-8")\n    return paths\n\n\ndef write_outputs(*, figures: bool = True) -> dict:\n    """Generate the scoped CSV/JSON/TeX artifacts and optional publication figure."""\n    data = load_focused_data()\n    OUTPUTS.mkdir(parents=True, exist_ok=True)\n    _write_csv(OUTPUTS / "Eligibility.csv", data["eligibility"])\n    _write_csv(OUTPUTS / "Completed_Peak_Summary.csv", data["summary"])\n    _write_csv(OUTPUTS / "Completed_Original_Peak_Results.csv", data["original"])\n    # Preserve the meaning of older machine-readable Eligible_* filenames.\n    _write_csv(OUTPUTS / "Eligible_Peak_Summary.csv", data["converged_subset"]["summary"])\n    _write_csv(OUTPUTS / "Eligible_Original_Peak_Results.csv", data["converged_subset"]["original"])\n    _write_csv(OUTPUTS / "Converged_Subset_Peak_Summary.csv", data["converged_subset"]["summary"])\n    _write_csv(OUTPUTS / "Converged_Subset_Original_Peak_Results.csv", data["converged_subset"]["original"])\n    (OUTPUTS / "Key_Results.json").write_text(json.dumps(data["key"], indent=2) + "\\n", encoding="utf-8")\n    (OUTPUTS / "Focused_Original_Peak_Table.tex").write_text(original_table(data), encoding="utf-8")\n    (OUTPUTS / "Focused_Repeated_Seed_Table.tex").write_text(repeated_table(data), encoding="utf-8")\n    (OUTPUTS / "Focused_Eligibility_Table.tex").write_text(eligibility_table(data), encoding="utf-8")\n    if figures:\n        make_figure(data)\n    return data["key"]\n\n\ndef generate_outputs(*, figures: bool = True) -> dict:\n    """Stable editorial API; write the requested focused assets from saved CSVs."""\n    return write_outputs(figures=figures)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--no-figures", action="store_true", help="Write only cached-score CSV/JSON/TeX summaries.")\n    args = parser.parse_args()\n    result = write_outputs(figures=not args.no_figures)\n    print(json.dumps({"OutputDirectory": str(OUTPUTS), "IncludedCountries": result["IncludedCountries"],\n                      "RepeatedRuns": result["RepeatedRuns"], "Comparisons": result["Comparisons"]}, indent=2))\n', 'residual_forecast_figures': '"""Restore both cached native filter trajectories in the focused forecast figure.\n\nThis module reads saved MATLAB CSVs only. It performs no model evaluation,\nfitting, resampling, trajectory repair, or selection by held-out performance.\nThe observed-scale main figure explicitly clips display values; the companion\npreserves both means and the full held-out intervals and member extrema.\nTraining fits are lines only; uncertainty is displayed for forecasts only.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nfrom datetime import datetime, timedelta\nimport hashlib\nimport sys\nimport json\nimport math\nfrom pathlib import Path\n\n\nROOT = Path(PACKAGE_ROOT)\nEDITORIAL = ROOT / "Data/Provenance"\nOUTPUTS = ROOT / "Results/Forecast"\nMAIN_STEM = "Restored_EnKF_EnKPF_Forecasts_Main"\nFULL_STEM = "Restored_EnKF_EnKPF_Forecasts_FullRange_S7"\nCOUNTRIES = ("Korea", "Japan", "Pakistan")\nOUTCOMES = ("Q", "R")\nMETHODS = ("EnKF", "EnKPF")\nSEEDS = {"Korea": 1, "Japan": 123, "Pakistan": 1}\nCOLORS = {"EnKF": "#1f77b4", "EnKPF": "#c44b20"}\nSIZE_INCHES = (7.0, 8.1)\nDPI = 600\nLINEAR_OBSERVED_MULTIPLIER = 1.20\nSYMLOG_THRESHOLD = 100.0\n\nMAIN_CAPTION = (\n    "Observed-scale fitting and forecast trajectories for the completed original "\n    "Korean, Japanese, and Pakistani peak origins. Black points are observations; blue EnKF "\n    "and orange EnKPF lines show training fits (solid) and forecasts (dashed), "\n    "with the vertical dashed line marking the final training date. Shading is "\n    "each method\'s central 95% empirical latent ensemble interval during the "\n    "held-out forecast period only; training fits are lines without shading. "\n    "Future measurement noise is excluded. For readable fitting "\n    "comparisons, each linear axis spans zero to 1.2 times the maximum observed "\n    "count. EnKF forecast means in the Japanese and Pakistani panels and interval tails in "\n    "several panels extend above these limits. Blue upward triangles mark the "\n    "first cached EnKF forecast date outside the displayed range. Figure S7 "\n    "shows the complete means, intervals, and member ranges without clipping. "\n    "Both filters use 200 members; random-number settings are documented in the supplementary methods, and "\n    "no realization was selected according to forecast error. Pakistan\'s LSM "\n    "initializer reached its evaluation budget (exit flag 0), although its "\n    "forecast completed; the converged-initializer subset is reported separately."\n)\n\nFULL_CAPTION = (\n    "Complete ranges of the same original Korean, Japanese, and Pakistani peak-origin "\n    "EnKF and EnKPF fitting and forecast trajectories shown in Figure 2. "\n    "Symmetric logarithmic axes, with a linear threshold of 100 counts, include "\n    "all observations and method means, along with the complete held-out latent "\n    "95% intervals and ensemble minima and maxima. Thin dotted curves show "\n    "method-specific forecast-member ranges; colored shading shows central "\n    "95% latent forecast intervals. Solid lines show training fits without "\n    "shading or member envelopes, dashed lines show forecasts, "\n    "and vertical gray lines mark the final training dates. No means, interval "\n    "endpoints, or member extrema are clipped or repaired. The latent ensemble "\n    "summaries exclude future measurement noise. Pakistan\'s training initializer "\n    "has exit flag 0 after reaching the evaluation budget; this completed "\n    "forecast is retained with that limitation disclosed."\n)\n\n\ndef _sha(path: Path) -> str:\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef _focused_helper():\n    return sys.modules["focused_peak_outputs"]\n\n\ndef _load_series() -> tuple[dict, dict]:\n    focused = _focused_helper()\n    data = focused.load_focused_data()\n    assert tuple(data["included"]) == COUNTRIES\n    source = focused.RESULTS / "script_reproduction" / "Forecast_Predictions.csv"\n    manifest_path = source.with_name("Run_Manifest.csv")\n    predictions = focused._read(source)\n    manifest = focused._read(manifest_path)\n    series = {}\n    for country in COUNTRIES:\n        runs = [row for row in manifest if row["Country"] == country]\n        assert len(runs) == 1 and runs[0]["Status"] == "success"\n        run = runs[0]\n        assert int(run["Seed"]) == SEEDS[country]\n        train, test = int(run["TrainDays"]), int(run["TestDays"])\n        for outcome in OUTCOMES:\n            common = None\n            for method in METHODS:\n                rows = sorted(\n                    (row for row in predictions if row["RunID"] == run["RunID"]\n                     and row["Outcome"] == outcome and row["Method"] == method),\n                    key=lambda row: int(row["ElapsedDay"]),\n                )\n                assert len(rows) == train + test\n                assert [int(row["ElapsedDay"]) for row in rows] == list(range(train + test))\n                assert [row["Period"] for row in rows] == ["Train"] * train + ["Test"] * test\n                assert {row["ProtocolID"] for row in rows} == {focused.PROTOCOL}\n                assert {row["ImplementationHash"] for row in rows} == {focused.IMPLEMENTATION}\n                dates = [datetime.fromisoformat(row["Date"]) for row in rows]\n                assert all(b - a == timedelta(days=1) for a, b in zip(dates, dates[1:]))\n                values = {\n                    field: [focused._num(row, field) for row in rows]\n                    for field in ("Observed", "Mean", "Low95", "High95", "MemberMin", "MemberMax")\n                }\n                assert all(value >= 0 for field in values.values() for value in field)\n                for i in range(train + test):\n                    assert (values["MemberMin"][i] <= values["Low95"][i]\n                            <= values["High95"][i] <= values["MemberMax"][i])\n                    assert values["MemberMin"][i] <= values["Mean"][i] <= values["MemberMax"][i]\n                if common is None:\n                    common = (dates, values["Observed"])\n                else:\n                    assert common == (dates, values["Observed"])\n                series[(country, outcome, method)] = {\n                    "RunID": run["RunID"], "Seed": SEEDS[country], "TrainDays": train,\n                    "TestDays": test, "Dates": dates, **values,\n                }\n    provenance = {\n        "ProtocolID": focused.PROTOCOL,\n        "ImplementationHash": focused.IMPLEMENTATION,\n        "Realizations": SEEDS,\n        "Countries": list(COUNTRIES),\n        "Methods": list(METHODS),\n        "SelectedByForecastError": False,\n        "EligibilityRule": data["key"]["EligibilityRule"],\n        "EligibilityChronology": data["key"]["EligibilityChronology"],\n        "UncertaintyTarget": "Central 95% latent ensemble quantiles; no future measurement noise.",\n        "TrainingUncertaintyDisplay": "None; training analysis means are lines only.",\n        "ForecastUncertaintyDisplay": "Central 2.5th--97.5th percentiles on held-out dates only; member minima/maxima also shown for held-out dates in S7.",\n        "BandBeginsAt": "First held-out observation date; no training date is included in a shaded polygon.",\n        "TrainingMeans": "Exact saved analysis means for each method; no interpolation or repair.",\n        "ForecastMeans": "Exact saved pure-forecast means; no filtering after the origin.",\n        "Sources": data["key"]["Sources"],\n        "HelperSHA256": hashlib.sha256(sys.modules["focused_peak_outputs"].__embedded_source__.encode("utf-8")).hexdigest(),\n        "GeneratorSHA256": hashlib.sha256(__embedded_source__.encode("utf-8")).hexdigest(),\n        "PredictionCSVSHA256": _sha(source),\n        "ManifestSHA256": _sha(manifest_path),\n    }\n    return series, provenance\n\n\ndef _outside(values, lower: float, upper: float) -> int:\n    return sum(value < lower or value > upper for value in values)\n\n\ndef _audit_rows(series: dict, *, full_range: bool, country: str, outcome: str,\n                lower: float, upper: float) -> list[dict]:\n    rows = []\n    for method in METHODS:\n        item = series[(country, outcome, method)]\n        train = item["TrainDays"]\n        for period, start, end in (("Train", 0, train), ("Test", train, len(item["Dates"]))):\n            means_outside = [i for i in range(start, end) if not lower <= item["Mean"][i] <= upper]\n            band_shown = period == "Test"\n            extrema_shown = full_range and period == "Test"\n            row = {\n                "Figure": "FullRange_S7" if full_range else "Main_ObservedScale",\n                "Country": country, "Outcome": outcome, "Method": method,\n                "RunID": item["RunID"], "Seed": item["Seed"], "Period": period,\n                "NPoints": end - start, "AxisScale": "symlog" if full_range else "linear",\n                "AxisMinimum": lower, "AxisMaximum": upper,\n                "IntervalShown": band_shown,\n                "MemberExtremaShown": extrema_shown,\n                "ObservedOutside": _outside(item["Observed"][start:end], lower, upper),\n                "MeanOutside": len(means_outside),\n                "Low95Outside": _outside(item["Low95"][start:end], lower, upper) if band_shown else 0,\n                "High95Outside": _outside(item["High95"][start:end], lower, upper) if band_shown else 0,\n                "MemberMinOutside": _outside(item["MemberMin"][start:end], lower, upper) if extrema_shown else 0,\n                "MemberMaxOutside": _outside(item["MemberMax"][start:end], lower, upper) if extrema_shown else 0,\n                "FirstMeanOutsideDate": item["Dates"][means_outside[0]].date().isoformat() if means_outside else "",\n                "LastMeanOutsideDate": item["Dates"][means_outside[-1]].date().isoformat() if means_outside else "",\n                "MaximumObserved": max(item["Observed"][start:end]),\n                "MaximumMean": max(item["Mean"][start:end]),\n                "MaximumHigh95": max(item["High95"][start:end]),\n                "MaximumMember": max(item["MemberMax"][start:end]),\n                "DisplayClippingDisclosedInCaption": not full_range,\n            }\n            if full_range:\n                assert not any(row[key] for key in (\n                    "ObservedOutside", "MeanOutside", "Low95Outside", "High95Outside",\n                    "MemberMinOutside", "MemberMaxOutside"))\n            rows.append(row)\n    return rows\n\n\ndef _make_figure(series: dict, *, full_range: bool) -> tuple[dict[str, Path], list[dict], list[dict]]:\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.dates as mdates\n    import matplotlib.pyplot as plt\n    from matplotlib.lines import Line2D\n    from matplotlib.patches import Patch\n    from matplotlib.ticker import FixedLocator, FuncFormatter, MaxNLocator\n    import numpy as np\n    from PIL import Image\n\n    rc = {\n        "font.family": "DejaVu Sans", "font.size": 10.5, "axes.labelsize": 10.5,\n        "axes.titlesize": 10.5, "xtick.labelsize": 10.5, "ytick.labelsize": 10.5,\n        "legend.fontsize": 10.5, "pdf.fonttype": 42, "ps.fonttype": 42,\n        "axes.spines.top": False, "axes.spines.right": False,\n        "mathtext.default": "regular",\n    }\n    audits, exit_markers = [], []\n    with plt.rc_context(rc):\n        fig, axes = plt.subplots(len(COUNTRIES), 2, figsize=SIZE_INCHES, squeeze=False)\n        fig.subplots_adjust(left=.135, right=.95, top=.965, bottom=.19, hspace=.51, wspace=.48)\n        for row, country in enumerate(COUNTRIES):\n            for col, outcome in enumerate(OUTCOMES):\n                ax = axes[row, col]\n                shared = series[(country, outcome, "EnKF")]\n                dates = np.array(shared["Dates"])\n                train = shared["TrainDays"]\n                observed = np.asarray(shared["Observed"])\n                if full_range:\n                    plotted_values = []\n                    for method in METHODS:\n                        item = series[(country, outcome, method)]\n                        for field in ("Observed", "Mean"):\n                            plotted_values.extend(item[field])\n                        for field in ("Low95", "High95", "MemberMin", "MemberMax"):\n                            plotted_values.extend(item[field][train:])\n                    maximum = max(plotted_values)\n                    upper = max(1., maximum) * 1.08\n                    ax.set_yscale("symlog", base=10, linthresh=SYMLOG_THRESHOLD, linscale=.8)\n                    powers = range(2, math.floor(math.log10(upper)) + 1)\n                    if len(powers) > 4:\n                        powers = range(2, math.floor(math.log10(upper)) + 1, 2)\n                    ax.yaxis.set_major_locator(FixedLocator([0.] + [10. ** exponent for exponent in powers]))\n                    ax.yaxis.set_major_formatter(FuncFormatter(\n                        lambda value, _: "0" if value == 0 else rf"$10^{{{int(round(math.log10(value)))}}}$"))\n                    ax.minorticks_off()\n                else:\n                    upper = max(1., float(observed.max())) * LINEAR_OBSERVED_MULTIPLIER\n                    ax.yaxis.set_major_locator(MaxNLocator(nbins=3, min_n_ticks=2, integer=True))\n                    ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:,.0f}"))\n                lower = 0.\n                for method in METHODS:\n                    item = series[(country, outcome, method)]\n                    color = COLORS[method]\n                    fitted = np.asarray(item["Mean"])\n                    lo, hi = np.asarray(item["Low95"]), np.asarray(item["High95"])\n                    if full_range:\n                        for bound in ("MemberMin", "MemberMax"):\n                            ax.plot(dates[train:], item[bound][train:], color=color,\n                                    lw=.65, ls=(0, (1, 2)), alpha=.65, zorder=1)\n                    ax.fill_between(dates[train:], lo[train:], hi[train:],\n                                    color=color, alpha=.18, linewidth=0, zorder=1)\n                    ax.plot(dates[:train], fitted[:train], color=color, lw=1.35, zorder=3)\n                    ax.plot(dates[train - 1:], fitted[train - 1:], color=color, lw=1.6,\n                            ls=(0, (4, 2.5)), zorder=3)\n                    if not full_range:\n                        beyond = np.flatnonzero(fitted[train:] > upper)\n                        if beyond.size:\n                            index = train + int(beyond[0])\n                            ax.plot(dates[index], upper, color=color, marker="^", markersize=5.0,\n                                    markeredgewidth=.7, lw=0, clip_on=False, zorder=6)\n                            exit_markers.append({\n                                "Country": country, "Outcome": outcome, "Method": method,\n                                "Date": item["Dates"][index].date().isoformat(),\n                                "UnmodifiedCachedMean": float(fitted[index]),\n                                "MarkerAtDisplayBoundary": upper,\n                                "NoInterpolatedCrossingDate": True,\n                            })\n                ax.plot(dates, observed, color="#202020", marker=".", lw=0, markersize=2.2, zorder=5)\n                ax.axvline(dates[train - 1], color="#777777", ls=(0, (2, 2)), lw=.9, zorder=2)\n                ax.set_ylim(lower, upper)\n                ax.set_xlim(dates[0], dates[-1])\n                ax.xaxis.set_major_locator(FixedLocator(mdates.date2num(\n                    [dates[0], dates[len(dates) // 2], dates[-1]])))\n                ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))\n                ax.set_ylabel("Active cases" if outcome == "Q" else "Recovered cases")\n                if row == len(COUNTRIES) - 1:\n                    ax.set_xlabel("Reporting date")\n                ax.set_title(f"{chr(65 + row * 2 + col)}  {country}: {outcome}",\n                             loc="left", fontweight="bold", pad=7)\n                ax.grid(axis="y", color="#d9dee5", linewidth=.65)\n                ax.set_axisbelow(True)\n                audits.extend(_audit_rows(series, full_range=full_range, country=country,\n                                          outcome=outcome, lower=lower, upper=upper))\n\n        # Three aligned legend columns keep the two methods\' phase and band\n        # definitions together at the final seven-inch publication width.\n        common = [\n            Line2D([], [], color="#202020", marker=".", lw=0, markersize=4, label="Observed"),\n            Line2D([], [], color="#777777", ls=(0, (2, 2)), lw=.9, label="Forecast origin"),\n            Line2D([], [], color="#777777", ls=(0, (1, 2)), lw=.8, label="Forecast range") if full_range\n            else Line2D([], [], lw=0, color="white", label=" "),\n        ]\n        handles = list(common)\n        for method in METHODS:\n            handles.extend([\n                Line2D([], [], color=COLORS[method], lw=1.35, label=f"{method} fit"),\n                Line2D([], [], color=COLORS[method], lw=1.6, ls=(0, (4, 2.5)), label=f"{method} forecast"),\n                Patch(facecolor=COLORS[method], alpha=.18, edgecolor="none", label=f"{method} forecast 95%"),\n            ])\n        fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(.51, .019), ncol=3,\n                   frameon=False, columnspacing=1.2, handlelength=2.1, handletextpad=.5,\n                   labelspacing=.55)\n        stem = FULL_STEM if full_range else MAIN_STEM\n        paths = {kind: ROOT / ("Figures/TIFF" if kind == "TIFF" else "Figures/PNG") / f"{stem}.{extension}" for kind, extension in\n                 (("PDF", "pdf"), ("PNG", "png"), ("TIFF", "tif"))}\n        fig.savefig(paths["PNG"], dpi=DPI, facecolor="white")\n        fig.savefig(paths["PDF"], facecolor="white", metadata={\n            "Title": "EnKF and EnKPF original peak-origin fitting and forecasts" +\n                     (": complete ensemble ranges" if full_range else ": observed-scale view"),\n            "Subject": "Cached native MATLAB results; latent ensemble uncertainty; no new simulation",\n        })\n        plt.close(fig)\n    with Image.open(paths["PNG"]) as raster:\n        assert raster.size == (round(SIZE_INCHES[0] * DPI), round(SIZE_INCHES[1] * DPI))\n        raster.convert("RGB").save(paths["TIFF"], compression="tiff_lzw", dpi=(DPI, DPI))\n    return paths, audits, exit_markers\n\n\ndef generate_forecast_figures() -> dict:\n    """Create both publication views and an explicit clipping/provenance audit."""\n    OUTPUTS.mkdir(parents=True, exist_ok=True)\n    series, provenance = _load_series()\n    main, main_audit, markers = _make_figure(series, full_range=False)\n    full, full_audit, _ = _make_figure(series, full_range=True)\n    audit = main_audit + full_audit\n    audit_path = OUTPUTS / "Clipping_Audit.csv"\n    with audit_path.open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(audit[0]))\n        writer.writeheader()\n        writer.writerows(audit)\n    captions_path = OUTPUTS / "Forecast_Figure_Captions.txt"\n    captions_path.write_text(f"Figure 2. {MAIN_CAPTION}\\n\\nFigure S7. {FULL_CAPTION}\\n", encoding="utf-8")\n    provenance.update({\n        "FigureSizeInches": list(SIZE_INCHES), "RasterDPI": DPI,\n        "Font": "DejaVu Sans", "FontSizePoints": 10.5, "Colors": COLORS,\n        "MainDisplay": {\n            "AxisScale": "linear", "AxisRule": "[0, 1.2 * maximum observed count] in every panel",\n            "IntentionalDisplayClipping": True,\n            "ClippingDisclosure": MAIN_CAPTION,\n            "ExitMarkers": markers,\n            "MemberExtremaShown": False,\n        },\n        "CompanionDisplay": {\n            "AxisScale": "symlog", "LinearThresholdCounts": SYMLOG_THRESHOLD,\n            "AxisRule": "[0, 1.08 * maximum of all observed, means, intervals, and member extrema]",\n            "IntentionalDisplayClipping": False, "MemberExtremaShown": True,\n        },\n        "ClippingAudit": audit,\n        "Files": {path.name: _sha(path) for path in (*main.values(), *full.values(), audit_path, captions_path)},\n    })\n    provenance_path = OUTPUTS / "Forecast_Figure_Provenance.json"\n    provenance_path.write_text(json.dumps(provenance, indent=2) + "\\n", encoding="utf-8")\n    return {\n        "main": main, "full_range": full, "audit": audit_path, "audit_rows": audit,\n        "provenance": provenance_path, "captions": captions_path,\n        "main_caption": MAIN_CAPTION, "full_range_caption": FULL_CAPTION,\n    }\n\n\nif __name__ == "__main__":\n    result = generate_forecast_figures()\n    print(json.dumps({\n        "Main": {kind: str(path) for kind, path in result["main"].items()},\n        "FullRange": {kind: str(path) for kind, path in result["full_range"].items()},\n        "Audit": str(result["audit"]), "Provenance": str(result["provenance"]),\n    }, indent=2))\n'}

for module_name, module_source in EMBEDDED_MODULE_SOURCES.items():
    embedded_module = types.ModuleType(module_name)
    embedded_module.__file__ = f"<embedded publication notebook: {module_name}>"
    embedded_module.PACKAGE_ROOT = PROJECT_ROOT
    embedded_module.__embedded_source__ = module_source
    sys.modules[module_name] = embedded_module
    exec(compile(module_source, embedded_module.__file__, "exec"), embedded_module.__dict__)

COUNTRY_ORDER = getattr(sys.modules['manuscript_publication'], 'COUNTRY_ORDER')
DATE_CONFIG = getattr(sys.modules['manuscript_publication'], 'DATE_CONFIG')
load_publication_data = getattr(sys.modules['manuscript_publication'], 'load_publication_data')
behavioral_change_table = getattr(sys.modules['manuscript_publication'], 'behavioral_change_table')
plot_fitted_trajectories = getattr(sys.modules['manuscript_publication'], 'plot_fitted_trajectories')
plot_behavioral_scenarios = getattr(sys.modules['manuscript_publication'], 'plot_behavioral_scenarios')
plot_parameter_overview = getattr(sys.modules['manuscript_publication'], 'plot_parameter_overview')
plot_within_wave_changes = getattr(sys.modules['manuscript_publication'], 'plot_within_wave_changes')
plot_parameter_comparison = getattr(sys.modules['manuscript_publication'], 'plot_parameter_comparison')
export_publication_tables = getattr(sys.modules['manuscript_table_exports'], 'export_publication_tables')
current_endpoint_diagnostics = getattr(sys.modules['publication_diagnostics'], 'current_endpoint_diagnostics')
refresh_gaussianity_figures = getattr(sys.modules['publication_diagnostics'], 'refresh_gaussianity_figures')
save_publication_figure = getattr(sys.modules['publication_styles'], 'save_publication_figure')
style_publication_table = getattr(sys.modules['amm_publication_outputs'], 'style_publication_table')
cached_table_paths = getattr(sys.modules['amm_publication_outputs'], 'cached_table_paths')
focused_peak_module = sys.modules["focused_peak_outputs"]
FIGURES = [('Main', '1', 'Joint active/recovered fitting', 'Combined_3x2_Panel_Fitting'), ('Main', '2', 'Original EnKF and EnKPF peak forecasts', 'Restored_EnKF_EnKPF_Forecasts_Main'), ('Main', '3', 'Completed idealized parameter recovery', 'AMM_Synthetic_Parameter_Recovery'), ('Main', '4', 'Behavioral parameters and timed scenarios', 'graphical_abstract'), ('Main', '5', 'Date-resolved EnKPF behavioral parameters', 'Figure3_EnKPF_Behavioral_Parameter_Trajectories'), ('Main', '6', 'Within-wave EnKPF parameter changes', 'Figure4_Behavioral_Parameters_By_Wave_Country_PercentChange'), ('Main', '7', 'Common-design LHS PRCC', 'Figure_Global_LHS_PRCC_QR'), ('Supplementary', 'S1', 'Endpoint JB and Mardia p-values', 'Supplementary_Figure_1_p_values'), ('Supplementary', 'S2', 'Endpoint marginal skewness', 'Supplementary_Figure_2_Skewness'), ('Supplementary', 'S3', 'Endpoint marginal excess kurtosis', 'Supplementary_Figure_3_Kurtosis'), ('Supplementary', 'S4', 'Korea parameter comparisons', 'Parameter_Comparison_South_Korea_WaveWise'), ('Supplementary', 'S5', 'Japan parameter comparisons', 'Parameter_Comparison_Japan_WaveWise'), ('Supplementary', 'S6', 'Pakistan parameter comparisons', 'Parameter_Comparison_Pakistan_WaveWise'), ('Supplementary', 'S7', 'Complete EnKF and EnKPF forecast ranges', 'Restored_EnKF_EnKPF_Forecasts_FullRange_S7')]
print(f"Portable publication package: {PROJECT_ROOT.name}")


## 3. Settings

In [ ]:
FIGURE_DPI = 600
N_LHS = 3000
LHS_SEED = 2026

# Cached review is the default; no numerical simulations run in this notebook.
REBUILD_EXISTING_FIGURES = False
EXPORT_NUMBERED_TABLES = True

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 9, "mathtext.fontset": "dejavusans",
    "axes.titlesize": 10, "axes.labelsize": 9, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "figure.facecolor": "white", "savefig.facecolor": "white", "pdf.fonttype": 42,
})

def save_figure(fig, stem, dpi=FIGURE_DPI):
    return save_publication_figure(fig, stem, PNG_DIR, TIFF_DIR, dpi=dpi)

def show_figure(fig, stem):
    path = save_figure(fig, stem)
    display(NotebookImage(filename=str(path), width=950))
    plt.close(fig)

def saved_figure_path(stem: str, suffix: str = ".png") -> Path:
    for directory in (PNG_DIR, TIFF_DIR):
        path = directory / f"{stem}{suffix}"
        if path.is_file():
            return path
    raise FileNotFoundError(f"Required reviewed figure is missing: {stem}{suffix}")

def show_saved(stem: str):
    display(NotebookImage(filename=str(saved_figure_path(stem)), width=950))

def show_table(stem):
    item = publication_tables[stem]
    number = stem.rsplit("_", 1)[1]
    section = "Main" if stem.startswith("Main") else "Supplementary"
    display(Markdown(f"### {section} Table {number}: {item['caption']}"))
    frame = item.get("display_data", item["data"])
    display(style_publication_table(item))
    display(Markdown(item["notes"]))

## 4. Load current fitting, parameter and policy results
Dates and observations are matched to the original country Excel windows, including shared wave boundaries. Policy summaries must use the same fitted parameters as the hybrid workbooks.

In [ ]:
data = load_publication_data(PROJECT_ROOT, COUNTRIES)
parameter_wide = data["parameter_wide"]
protocol = data["policy_protocol"]
display(Markdown("**Saved policy design:** " + protocol["description"] + " " + protocol["interpretation"]))
data["sources"].to_csv(TABLES_DIR / "Publication_Input_Sources.csv", index=False)
parameter_wide.to_csv(TABLES_DIR / "Publication_Parameter_Trajectories.csv", index=False)
parameter_changes = behavioral_change_table(parameter_wide)
parameter_changes.to_csv(TABLES_DIR / "Publication_Behavioral_Parameter_Changes.csv", index=False)
display(data["sources"][["Source", "Modified_UTC"]])

## 5. Cached Gaussianity diagnostics and forecast initializer status
Gaussianity uses the matched full-wave EnKF endpoint ensembles. Standalone PF has its own paired reference. All three completed peak forecasts are retained; optimizer flags define only the separate Korea/Japan robustness subset. Displayed metadata is read from saved results, not a new fitting or convergence run.


In [ ]:
jb = pd.read_csv(TABLES_DIR / "current_hybrid_EnKF_JB_endpoints.csv")
mardia = pd.read_csv(TABLES_DIR / "current_hybrid_EnKF_Mardia_endpoints.csv")
focused_peak_data = focused_peak_module.load_focused_data()
print(f"Endpoint JB rejections: {int(jb.JB_reject.sum())}/{len(jb)}")
print(f"Endpoint Mardia rejections: {int(mardia.Reject_Joint.sum())}/{len(mardia)}")
display(pd.DataFrame(focused_peak_data["eligibility"])[
    ["Country", "OriginalOptimizerExitFlag", "OriginalOptimizerEvaluations", "PrimaryIncluded", "ConvergedSubsetIncluded", "Reason"]
])


## 6. Cached common Latin hypercube sensitivity design

The reviewed common 3,000-point parameter design was simulated under nine country–wave conditions. This notebook reads its saved PRCC, bounds, window, and diagnostic CSV files. Responses are equal-weight mean terminal Q/N and R/N. Main Figure 7 is the reviewed PRCC asset.


In [ ]:
sensitivity = {
    key: pd.read_csv(TABLES_DIR / f"{stem}.csv")
    for key, stem in {
        "table": "global_LHS_PRCC", "bounds": "global_LHS_PRCC_parameter_bounds",
        "windows": "global_LHS_PRCC_wave_windows", "diagnostics": "global_LHS_PRCC_diagnostics",
    }.items()
}
display(sensitivity["bounds"])


## 7. Current manuscript and supplementary tables
The sequence matches Main Tables 1–4 and Supplementary Tables S1–S20. Main Table 3 compares all six original outcomes across Korea, Japan, and Pakistan. S17–S18 cover completed synthetic recovery and local information; S19 reports completion and initializer flags; S20 groups all-country and converged-initializer 14-day summaries.

All five declared seeds are retained at each included origin. Pakistan’s budget-limited initializer is disclosed in the primary comparison and omitted only from the convergence subset, uniformly across methods and seeds. No incomplete-extension or unresolved-profile table is added. Captions follow the canonical TeX sources. Set `EXPORT_NUMBERED_TABLES = True` to save the current workbook and CSV/TeX fragments.


In [ ]:
from collections import OrderedDict
import re

# Caption/label metadata follow the current canonical sources, not an older AMM map.
def canonical_table_metadata(path: Path) -> dict[str, str]:
    metadata = json.loads((PROJECT_ROOT / "Data/Provenance/Table_Captions.json").read_text(encoding="utf-8"))
    return metadata[path.stem]

main_captions = canonical_table_metadata(PYTHON_DIR / "Manuscript" / "Main_Manuscript.tex")
si_captions = canonical_table_metadata(PYTHON_DIR / "Manuscript" / "Supplementary_Manuscript.tex")
if len(main_captions) != 4 or len(si_captions) != 20:
    raise ValueError("Canonical table counts changed; reconcile the notebook numbering before exporting.")

cached_index = pd.read_csv(TABLES_DIR / "Manuscript_Supplementary_Tables_Index.csv").fillna("")
publication_tables = OrderedDict()

def add_current_table(key: str, frame: pd.DataFrame, label: str, notes: str,
                      csv_source: str, *, display_data: pd.DataFrame | None = None):
    metadata = main_captions if key.startswith("Main") else si_captions
    if label not in metadata or frame.empty:
        raise ValueError(f"Current manuscript table is missing or empty: {key}, {label}")
    frame = frame.copy()
    if "Country" in frame:
        frame["Country"] = frame["Country"].replace({"South Korea": "Korea"})
    item = {"data": frame, "caption": metadata[label], "label": label, "notes": notes,
            "sources": [csv_source], "csv_source": csv_source,
            "tex_source": f"Results/manuscript_tables/{key}.tex"}
    if display_data is not None:
        item["display_data"] = display_data
    publication_tables[key] = item

for key in ("Main_Table_1", "Main_Table_2"):
    metadata = cached_index.loc[cached_index.Output.eq(key)]
    if len(metadata) != 1:
        raise ValueError(f"The cached table index must contain exactly one {key}.")
    row = metadata.iloc[0]
    add_current_table(key, pd.read_csv(TABLES_DIR / f"{key}.csv"), str(row["Label"]),
                      str(row["Notes"]), f"Results/Tables/{key}.csv")

original_peak = pd.DataFrame(focused_peak_data["original"]).rename(columns={
    "RMSE_EnKF": "RMSE EnKF", "RMSE_EnKPF": "RMSE EnKPF",
    "R2_EnKF": "R2 EnKF", "R2_EnKPF": "R2 EnKPF",
    "EnKPF_LatentCoverage95Percent": "EnKPF latent coverage (%)",
})
original_peak = original_peak[["Country", "Outcome", "RMSE EnKF", "RMSE EnKPF", "R2 EnKF", "R2 EnKPF", "EnKPF latent coverage (%)"]].copy()
add_current_table("Main_Table_3", original_peak, "tab:focused-peak-results",
                  "Both filters use 200 members; random-number settings are given in Supplementary Table S11. All six completed outcome comparisons are retained. Coverage is latent, not measurement-inclusive. Pakistan's initializer budget stop is disclosed; the converged-initializer subset is reported separately.",
                  "Data/NativeForecast/script_reproduction/Forecast_Metrics.csv")
policy_source = "Data/Validation/summary/AMM_Policy_Table_4.csv"
add_current_table("Main_Table_4", pd.read_csv(PROJECT_ROOT / policy_source), "tab:policy",
                  "Equal +5% beta_f perturbations activate at 10% and 30% of elapsed wave duration. These are conditional simulations, not causal real-world policy estimates.", policy_source)

for number in range(1, 17):
    key = f"Supplementary_Table_S{number}"
    metadata = cached_index.loc[cached_index.Output.eq(key)]
    if len(metadata) != 1:
        raise ValueError(f"The cached table index must contain exactly one {key}.")
    row = metadata.iloc[0]
    frame = pd.read_csv(TABLES_DIR / f"{key}.csv")
    if "Country" in frame:
        frame["Country"] = frame["Country"].replace({"South Korea": "Korea"})
    notes = str(row["Notes"])
    if number == 1:
        notes = ("First-wave LSM starting guesses; later-wave LSM starts use the preceding LSM estimates. "
                 "Korean PLE starts from the same wave LSM fit; Japanese and Pakistani first-wave PLE start from LSM, "
                 "and later-wave PLE starts from the preceding PLE fit. Filter centers are wave-specific PLE estimates.")
    elif number == 2:
        first_wave = frame["Wave"].eq(1)
        frame.loc[first_wave, "S0"] = "N0 - Q0 - R0 - fitted I0, Wave 1"
        frame.loc[first_wave, "I0"] = "Fitted I0, Wave 1"
        notes = ("Q0 and R0 are the first observed counts of each wave. I0 is estimated jointly with the rates; "
                 "for Wave 1, S0 is recomputed as N0-Q0-R0-fitted I0. Each initial ensemble member uses its sampled I0 before physical projection. "
                 "Optimizer starting guesses appear separately in Table S1.")
    elif number in (11, 12, 13):
        frame = frame.loc[frame["Country"].isin(focused_peak_data["included"])].copy()
        notes = ("All three completed original peak forecasts are retained. Pakistan has a budget-limited initializer; Korea/Japan forms the separate convergence subset. "
                 "No forecast row is selected by its held-out error. Intervals are latent ensemble quantiles; the residual score is descriptive.")
    elif number == 14:
        peak_row = frame["Experiment"].astype(str).str.contains("peak", case=False)
        frame.loc[peak_row, "Initializer and target"] = "Training-only LSM with termination flags disclosed; original and five-seed repetitions"
        frame.loc[peak_row, "Reported quantities"] = "Completed three-country peaks; separate converged Korea/Japan subset; fixed cutoff parameters; no future assimilation or process noise; latent interval evaluation"
    add_current_table(key, frame, str(row["Label"]), notes, f"Results/Tables/{key}.csv")

recovery_source = "Data/Validation/summary/Synthetic_Recovery_Compact.csv"
recovery = pd.read_csv(PROJECT_ROOT / recovery_source)
if not recovery["ConvergedSelectedFits"].eq(recovery["ExpectedReplicates"]).all():
    raise ValueError("The completed synthetic recovery source contains an unresolved selected fit.")
recovery = recovery[["Truth", "Target", "Parameter", "True", "MeanEstimate", "RelativeBiasPercent", "RelativeRMSEPercent"]].copy()
recovery["Target"] = recovery["Target"].replace({"Q_only": "Q only", "joint_QR": "Joint Q/R"})
add_current_table("Supplementary_Table_S17", recovery, "tab:amm-recovery",
                  "All 120 selected synthetic fits converge, including one boundary estimate; every selected fit is retained. This is an idealized model-correct design.", recovery_source)

information_source = "Data/Validation/synthetic_recovery/Synthetic_Unregularized_Information.csv"
information_raw = pd.read_csv(PROJECT_ROOT / information_source)
information = information_raw.groupby(["Truth", "Target"]).agg(
    Rank=("NumericalRank", "first"), Condition=("ConditionNumber", "first"),
    SmallestSingularValue=("WeightedLogJacobianSingularValue", "min"),
).reset_index()
information["Target"] = information["Target"].replace({"Q_only": "Q only", "joint_QR": "Joint Q/R"})
add_current_table("Supplementary_Table_S18", information, "tab:amm-information",
                  "Unregularized local information diagnostics; no unresolved profile figure or confidence interval is inferred.", information_source)

eligibility = pd.DataFrame(focused_peak_data["eligibility"])[["Country", "OriginalOptimizerExitFlag", "OriginalOptimizerEvaluations", "PrimaryIncluded", "ConvergedSubsetIncluded", "Reason"]]
add_current_table("Supplementary_Table_S19", eligibility, "tab:focused-eligibility",
                  focused_peak_data["key"]["EligibilityRule"] + " " + focused_peak_data["key"]["EligibilityChronology"],
                  "Data/NativeForecast/script_reproduction/Fit_Diagnostics.csv")
seed_frames = []
for scope, rows in (
    ("All completed origins (Korea, Japan, Pakistan)", focused_peak_data["summary"]),
    ("Converged-initializer subset (Korea, Japan)", focused_peak_data["converged_subset"]["summary"]),
):
    frame = pd.DataFrame(rows)
    frame = frame.loc[frame["Horizon"].eq("14-day"), ["Method", "MacroNRMSE", "SharedSeedMacroSD", "LatentCoverage95Percent"]].copy()
    frame.insert(0, "Scope", scope)
    seed_frames.append(frame)
seed_summary = pd.concat(seed_frames, ignore_index=True)
seed_display = seed_summary.copy()
seed_display["MacroNRMSE"] = [
    f"{value:.3f} ± {sd:.3f}" if method in ("EnKF", "EnKPF") else f"{value:.3f}"
    for method, value, sd in zip(seed_summary["Method"], seed_summary["MacroNRMSE"], seed_summary["SharedSeedMacroSD"])
]
seed_display = seed_display.drop(columns="SharedSeedMacroSD")
seed_display["LatentCoverage95Percent"] = seed_display["LatentCoverage95Percent"].map(lambda value: "—" if pd.isna(value) else f"{value:.1f}")
add_current_table("Supplementary_Table_S20", seed_summary, "tab:focused-seed-summary",
                  "Equal Q/R and all five declared seeds at each origin. Primary: three completed origins; robustness: converged Korea/Japan subset. SD concerns shared-seed macro means, not independent epidemics. Deterministic references have no interval coverage.",
                  "Data/NativeForecast/peak_stability/Forecast_Metrics.csv", display_data=seed_display)

if len(publication_tables) != 24:
    raise ValueError("The current publication scope requires four main and twenty supplementary tables.")
table_paths = cached_table_paths(PROJECT_ROOT, publication_tables)
if EXPORT_NUMBERED_TABLES:
    table_paths = export_publication_tables(publication_tables, TABLES_DIR, TEX_DIR)
print(f"Current tables: {len(publication_tables)} (4 main + 20 supplementary)")
print(f"Workbook export enabled: {EXPORT_NUMBERED_TABLES}")


### Main tables

In [ ]:
for number in range(1, 5):
    show_table(f"Main_Table_{number}")


### Supplementary tables

In [ ]:
for number in range(1, 21):
    show_table(f"Supplementary_Table_S{number}")


# Main manuscript figures

## Figure 1. Joint active/recovered fitting
Shading is the exported ensemble mean ±1.96 ensemble SD. These Gaussian moment bands are distinct from forecast percentile intervals.

In [ ]:
if REBUILD_EXISTING_FIGURES:
    show_figure(plot_fitted_trajectories(data), "Combined_3x2_Panel_Fitting")
else:
    show_saved("Combined_3x2_Panel_Fitting")


## Figure 2. Original EnKF and EnKPF peak forecasts
Both methods' saved training fits, held-out means, and latent 95% intervals are restored for the original eligible Korea/Japan realizations. The main linear view uses an observed-scale limit for readability. Off-scale forecasts and interval tails remain documented in the display audit; Figure S7 retains the complete ranges without clipping. No model is run and no seed is selected by its forecast error.

Training fits are solid mean lines only. Shading is the central empirical 95% ensemble interval on held-out dates only, starting with the first forecast observation. In S7, dotted member ranges are also forecast-only. Figure 1 separately retains full-wave mean ±1.96 SD bands.


In [ ]:
restored_forecast_api = sys.modules["residual_forecast_figures"]
restored_peak_exports: dict[str, Any] = restored_forecast_api.generate_forecast_figures()
display(Markdown(restored_peak_exports["main_caption"]))
display(NotebookImage(filename=str(restored_peak_exports["main"]["PNG"]), width=1000))


## Figure 3. Idealized synthetic parameter recovery
Three generating vectors and two observation targets each use 20 paired Poisson replicates. All selected fits are retained. Larger behavioral-rate errors persist in Truths 2 and 3. These static recovery experiments are separate from the filtering benchmark.


In [ ]:
show_saved("AMM_Synthetic_Parameter_Recovery")


## Figure 4. Behavioral parameters and timed scenarios
Panels A/B show wave-time mean behavioral parameters. Panel C compares mean peak reductions for Early, Late, and No change; Panel D shows early-minus-late differences by wave. Perturbation magnitudes and activation fractions are read from the saved MATLAB exports and reported below. Equal-magnitude scenarios compare activation timing within the fitted model. If activation follows the baseline peak, the whole-wave peak reduction may be zero despite later trajectory changes. These are conditional simulations, not causal policy-effect estimates.

In [ ]:
if REBUILD_EXISTING_FIGURES:
    display(Markdown(data["policy_protocol"]["description"] + " " + data["policy_protocol"]["interpretation"]))
    fig2 = plot_behavioral_scenarios(data)
    # Keep these display labels explicit even if only this cell is rerun.
    fig2.axes[2].set_xticks([0, 1, 2], ["Early", "Late", "No change"])
    fig2.axes[3].set_yticks(np.arange(len(COUNTRY_ORDER)) + .5,
                          ["Korea" if country == "South Korea" else country for country in COUNTRY_ORDER])
    for axis in fig2.axes[:2]:
        for line in axis.lines:
            if line.get_label() == "South Korea":
                line.set_label("Korea")
    show_figure(fig2, "graphical_abstract")
else:
    show_saved("graphical_abstract")


## Figure 5. Date-resolved EnKPF behavioral parameters
Country-specific dates and wave boundaries are retained. Wave-time summaries are effective fitted parameters rather than direct behavioral measurements.

In [ ]:
if REBUILD_EXISTING_FIGURES:
    show_figure(plot_parameter_overview(parameter_wide), "Figure3_EnKPF_Behavioral_Parameter_Trajectories")
else:
    show_saved("Figure3_EnKPF_Behavioral_Parameter_Trajectories")


## Figure 6. Within-wave parameter changes
Each annotation is the net first-to-last EnKPF change: $100(\theta_{end}-\theta_{start})/|\theta_{start}|$. Red indicates an increase and green a decrease. Countries are stacked vertically; the two parameter columns have additional horizontal spacing.

In [ ]:
if REBUILD_EXISTING_FIGURES:
    show_figure(plot_within_wave_changes(parameter_wide), "Figure4_Behavioral_Parameters_By_Wave_Country_PercentChange")
else:
    show_saved("Figure4_Behavioral_Parameters_By_Wave_Country_PercentChange")


## Figure 7. PRCC
Panels Q and R refer to the equal-weight mean terminal population-normalized outputs $Q/N$ and $R/N$ across the nine country-wave conditions. One common Latin hypercube design is used for all conditions. Stars indicate unadjusted residual-correlation tests: * p < 0.05, ** p < 0.01, *** p < 0.001. They do not test filter superiority.

In [ ]:
show_saved("Figure_Global_LHS_PRCC_QR")
display(sensitivity["table"])


# Supplementary figures

## Figures S1–S3. Current endpoint Gaussianity diagnostics
These are wave-end summaries from the matched hybrid-run EnKF ensembles. S3 shows **excess** kurtosis, whose Gaussian reference is zero. JB uses the finite-sample MATLAB convention at 200 members; p-value bounds are retained in the CSV.

In [ ]:
if REBUILD_EXISTING_FIGURES:
    jb, mardia = refresh_gaussianity_figures(
        PROJECT_ROOT, COUNTRIES, TABLES_DIR, TEX_DIR, PNG_DIR, TIFF_DIR,
    )

for label, stem in (
    ("Figure S1: JB and Mardia p-values", "Supplementary_Figure_1_p_values"),
    ("Figure S2: marginal skewness", "Supplementary_Figure_2_Skewness"),
    ("Figure S3: marginal excess kurtosis", "Supplementary_Figure_3_Kurtosis"),
):
    display(Markdown(f"### {label}"))
    show_saved(stem)

## Figure S4. Korea: EnKF and EnKPF parameter trajectories
Each wave uses its own reporting dates. Parameters are plotted separately, with no temporal concatenation across wave boundaries.


In [ ]:
if REBUILD_EXISTING_FIGURES:
    show_figure(plot_parameter_comparison(parameter_wide, 'South Korea'), 'Parameter_Comparison_South_Korea_WaveWise')
else:
    show_saved("Parameter_Comparison_South_Korea_WaveWise")


## Figure S5. Japan: EnKF and EnKPF parameter trajectories
Each wave uses its own reporting dates. Parameters are plotted separately, with no temporal concatenation across wave boundaries.

In [ ]:
if REBUILD_EXISTING_FIGURES:
    show_figure(plot_parameter_comparison(parameter_wide, 'Japan'), 'Parameter_Comparison_Japan_WaveWise')
else:
    show_saved("Parameter_Comparison_Japan_WaveWise")


## Figure S6. Pakistan: EnKF and EnKPF parameter trajectories
Each wave uses its own reporting dates. Parameters are plotted separately, with no temporal concatenation across wave boundaries.

In [ ]:
if REBUILD_EXISTING_FIGURES:
    show_figure(plot_parameter_comparison(parameter_wide, 'Pakistan'), 'Parameter_Comparison_Pakistan_WaveWise')
else:
    show_saved("Parameter_Comparison_Pakistan_WaveWise")


## Figure S7. Complete original EnKF and EnKPF forecast ranges
This companion uses the same three countries, original seeds, dates, observations, and saved arrays as Main Figure 2. Symmetric-log axes retain every mean, latent 95% interval endpoint, and ensemble minimum/maximum. These are model-ensemble intervals without future measurement noise. The companion prevents the observed-scale main view from concealing large forecast drift or wide bands.

Training fits are solid mean lines only. Shading is the central empirical 95% ensemble interval on held-out dates only, starting with the first forecast observation. In S7, dotted member ranges are also forecast-only. Figure 1 separately retains full-wave mean ±1.96 SD bands.


In [ ]:
s7_forecast_api = sys.modules["residual_forecast_figures"]
s7_peak_exports: dict[str, Any] = s7_forecast_api.generate_forecast_figures()
display(Markdown(s7_peak_exports["full_range_caption"]))
display(NotebookImage(filename=str(s7_peak_exports["full_range"]["PNG"]), width=1000))


## 8. Current output index and source record
The index includes only the current 14 numbered figures and 24 tables. It records source paths relative to the project, current native-source hashes, and reporting scope. Removed benchmark/profile displays and incomplete extensions are not restored by this notebook. Full numerical experiments remain available in their separate cached analysis records.


In [ ]:
import hashlib

index_rows = []
for section, number, title, stem in FIGURES:
    index_rows.append({
        "Section": section, "Reference": f"Figure {number}", "Title": title,
        "PNG": saved_figure_path(stem, ".png").relative_to(PROJECT_ROOT).as_posix(),
        "PDF": saved_figure_path(stem, ".pdf").relative_to(PROJECT_ROOT).as_posix(),
        "TIFF": saved_figure_path(stem, ".tif").relative_to(PROJECT_ROOT).as_posix(),
        "CSV": "", "TeX": "",
    })
for key, item in publication_tables.items():
    info = table_paths["tables"][key]
    index_rows.append({
        "Section": "Main" if key.startswith("Main") else "Supplementary",
        "Reference": "Table " + key.rsplit("_", 1)[1], "Title": item["caption"],
        "PNG": "", "PDF": "", "TIFF": "",
        "CSV": Path(info["csv"]).relative_to(PROJECT_ROOT).as_posix(),
        "TeX": Path(info["tex"]).relative_to(PROJECT_ROOT).as_posix(),
    })
portable_index = pd.DataFrame(index_rows)
portable_index.to_csv(TABLES_DIR / "Publication_Output_Index.csv", index=False)
canonical_paths = [PROJECT_ROOT / "Data/Provenance/Table_Captions.json"]
run_record = {
    "Scope": "4 main tables, 20 supplementary tables, 7 main figures, 7 supplementary figures",
    "NumericalModelRun": False,
    "ForecastCountries": list(focused_peak_data["included"]),
    "ForecastOriginalOutcomes": len(focused_peak_data["original"]),
    "ConvergedSubsetCountries": list(focused_peak_data["converged_subset"]["included"]),
    "ForecastComparisons": focused_peak_data["key"]["Comparisons"],
    "ForecastEligibility": focused_peak_data["key"]["EligibilityChronology"],
    "NativeSourceSHA256": focused_peak_data["key"]["Sources"],
    "CanonicalSourceSHA256": {path.relative_to(PROJECT_ROOT).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest() for path in canonical_paths},
    "JBEndpointRejections": int(jb.JB_reject.sum()), "MardiaEndpointRejections": int(mardia.Reject_Joint.sum()),
    "RebuildSavedResultPlots": REBUILD_EXISTING_FIGURES, "ExportNumberedTables": EXPORT_NUMBERED_TABLES,
}
manifest_path = TABLES_DIR / "Publication_Run_Manifest.json"
manifest_path.write_text(json.dumps(run_record, indent=2) + "\n", encoding="utf-8")
display(portable_index[["Section", "Reference", "Title"]])
print(f"Complete: {len(FIGURES)} figures and {len(publication_tables)} numbered tables.")
print("Numerical model reruns: none; figures and tables use cached results.")
print(f"Output index: {(TABLES_DIR / 'Publication_Output_Index.csv').relative_to(PROJECT_ROOT)}")


## Regenerate the restored peak-forecast figures from saved MATLAB exports

This final cell redraws **Main Figure 2 and Supplementary Figure S7** only. It runs no fitting, filtering, sampling, or forecast model. The two methods use the same saved original country dates, observations, seeds, and source hashes.

The primary scope is **six outcomes** (Korea/Japan/Pakistan × active/recovered). Pakistan's original training initializer reaches its evaluation budget (exit flag 0; 1,501 evaluations); its completed forecast is retained with this limitation disclosed. Korea/Japan is a separate converged-initializer robustness subset. The main observed-scale view can contain off-scale means or interval tails; its display audit records these explicitly. The full-range symmetric-log companion retains all means, intervals, and member extrema without clipping. These latent ensemble intervals exclude future measurement noise.

Training fits are solid mean lines only. Shading is the central empirical 95% ensemble interval on held-out dates only, starting with the first forecast observation. In S7, dotted member ranges are also forecast-only. Figure 1 separately retains full-wave mean ±1.96 SD bands.


In [ ]:
_native_plotter = sys.modules["residual_forecast_figures"]
regenerated_peak_exports: dict[str, Any] = _native_plotter.generate_forecast_figures()
for view, caption_key in (("main", "main_caption"), ("full_range", "full_range_caption")):
    display(Markdown(regenerated_peak_exports[caption_key]))
    display(NotebookImage(filename=str(regenerated_peak_exports[view]["PNG"]), width=1000))
print("Display audit:", Path(regenerated_peak_exports["audit"]).relative_to(PROJECT_ROOT).as_posix())
print("Numerical model reruns: none.")
